# PlantVision 2.2 — houseplant disease pipeline

End-to-end deterministic training notebook that produces every artefact
`server/` reads at runtime:

```
models/
├── classifier.pth              ← KD student weights (MobileNetV3-Large)
├── classifier.onnx             ← FP32 export (sanity / fallback)
├── classifier_int8.onnx        ← static INT8 QDQ (production)   [absent if INT8 drift > 0.05]
├── classifier_fp32.onnx        ← FP32 ONNX (only present if INT8 drift gate failed)
├── classes.json                ← {class_names, training.img_size, ood, kd, ...}
├── temperature.json            ← {temperature: T}
├── conformal.json              ← RAPS calibration {q_hat, alpha, ...}
├── detector.{pt,onnx}          ← Stage 1 (re-trained in §3 from PlantDoc bboxes by default)
├── training_metrics.json
└── model_card.json
```

## Class set — 8 trainable classes (v2.2)

```
gray_mold_botrytis, healthy, leaf_scorch_underwatering,
leaf_spot, not_a_plant, nutrient_deficiency,
powdery_mildew, viral_disease
```

`leaf_scorch_underwatering` and `nutrient_deficiency` are sourced from
free-text iNat queries (`leaf scorch`, `chlorosis`, …). They land at a
60-row per-class floor (`THRESH_TRAIN_PER_CLASS`); KD with logit
adjustment + label smoothing absorbs the noisier free-text labels.
Flip `INCLUDE_PHYSIOLOGICAL_CLASSES = False` in §1 to roll back to the
7-class head if those sources go dark.

The two remaining v2.0 design classes — `edema`, `sunburn_or_scorch` —
stay deferred. Their bilingual content cards live in
`server/diseases_data.py` already, so flipping them on is a §1 +
data-fetch change, not a content port.

## Why these decisions

* **Loss-side balancing only.** Logit Adjustment (Menon ICLR 2021,
  τ=0.5) is used; the `WeightedRandomSampler` is **off** by default to
  avoid double-correction on rare classes. A source-weighted sampler
  down-weights field-crop PlantDoc rows so the empirical training prior
  shifts toward indoor houseplants.
* **Three independent val slices, one ship gate.**
  `val_realworld` = primary disease macro-F1 gate (≥ 0.58 staged v1, no
  class with recall=0, healthy-bias < 1.5). `val_indomain` drives early
  stopping. `val_studio` (PlantVillage) is a transfer/background-bias
  monitor — never blocking.
* **Two dedicated calibration splits**, both held out from the train
  set: `temperature_cal` (~5 %) for INT8 calibration + temperature
  scaling, `conformal_cal` (~10 %) for RAPS quantile fitting. Both are
  restricted to the deployment distribution (`plantdoc + inat`),
  never PlantVillage, so conformal exchangeability holds at deploy.
* **EMA student** during KD (adaptive decay, half-life ≈ KD-steps/4)
  — stabilises the student in the small-data regime. The EMA copy is
  what we export.
* **Static INT8 (QDQ) ships only if `val_realworld` disease F1 drops
  by < 0.02** vs FP32. On drift > 0.05 the build does NOT silently
  rename FP32 as INT8 — it writes `classifier_fp32.onnx` and leaves
  `classifier_int8.onnx` absent, so the deploy bundle script can't
  mistake an FP32 graph for an INT8 one.
* **Fail-loud everywhere.** Every block raises `RuntimeError` on
  silent-failure paths (empty splits, taxonomy drift, label leakage,
  zero-recall classes, INT8 drift > 0.05 without explicit operator
  override).
* **Ship-gate-then-write.** All canonical artefacts in `models/` are
  written AFTER the ship gate passes. A failed run leaves the prior
  build untouched.

Total training time on a single A100-40GB is ≈ 45–60 min including
data fetch.


## §1 — Setup: deps, schema, taxonomy, eval helpers

This cell is idempotent — re-running it does not redo anything besides
imports and folder creation. It also:

* installs versioned deps,
* sets every seed,
* defines the `CropAnnotation` sidecar schema,
* declares the canonical `CLASSES` list (the alphabetical order is
  what `ImageFolder` will produce at training time and what every
  downstream artefact expects).


In [ ]:
# §1 — Setup
import os, sys, subprocess, random, json, hashlib, pathlib, re, shutil, time, zipfile, math
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Optional

# numpy is intentionally unpinned: recent torch / torchvision wheels link
# against numpy 2.x; pinning <2 breaks `from torchvision import …`.
PIP_PKGS = [
    'torch>=2.2', 'torchvision', 'timm>=1.0',
    'ultralytics>=8.2', 'onnx>=1.16', 'onnxruntime>=1.17',
    # torch>=2.10 imports onnxscript at torch.onnx.export module load.
    'onnxscript>=0.1',
    'imagehash', 'Pillow', 'scikit-learn>=1.4', 'pandas',
    'opencv-python-headless>=4.9', 'tqdm', 'matplotlib',
    'requests>=2.32', 'datasets>=2.18', 'PyYAML',
    'sympy',  # onnxruntime.quantization.shape_inference (quant_pre_process) needs it
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PIP_PKGS])

import numpy as np, requests, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms as T
from sklearn.metrics import f1_score, recall_score, confusion_matrix
from tqdm import tqdm

# ── Reproducibility ─────────────────────────────────────────────
SEED = 0
# Single source of truth for the version tag we write into
# ``classes.json`` and ``model_card.json``. Production servers pin
# behaviour to this string via ``classes.json -> model_version``.
MODEL_VERSION = '2.2'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True       # faster training
torch.backends.cudnn.deterministic = False  # honest about the trade-off

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device={device}  torch={torch.__version__}')

# ── Sidecar metadata schema ─────────────────────────────────────
@dataclass
class CropAnnotation:
    plant_id: str
    species: Optional[str]
    diagnosis: str
    severity: str               # low | medium | high | unknown
    leaf_or_whole_plant: str    # leaf | whole_plant
    capture_session: str
    source: str                 # inat | coco | plantvillage | plantdoc
    label_confidence: float     # in [0, 1]
    bbox: List[int]             # [x, y, w, h]; [0,0,0,0] for whole-image

def write_sidecar(p: Path, ann: CropAnnotation):
    p.with_suffix('.json').write_text(json.dumps(asdict(ann), ensure_ascii=False, indent=2))
def read_sidecar(p: Path):
    s = p.with_suffix('.json'); return json.loads(s.read_text()) if s.exists() else None

# ── Folder layout ───────────────────────────────────────────────
ROOT          = pathlib.Path('data_v2')
CLS_ROOT      = ROOT / 'classification'
TRAIN_ROOT    = CLS_ROOT / 'train'
VAL_ROOT      = CLS_ROOT / 'val'
VAL_INDOMAIN  = VAL_ROOT / 'indomain'
VAL_PRIMARY   = VAL_ROOT / 'realworld'              # ship gate
VAL_STUDIO    = VAL_ROOT / 'plantvillage_studio'    # transfer monitor
TEMP_CAL      = VAL_ROOT / 'temperature_cal'        # INT8 + temperature
CONFORMAL_CAL = VAL_ROOT / 'conformal_cal'          # RAPS

INCLUDE_PHYSIOLOGICAL_CLASSES = True
BASE_CLASSES = [
    'gray_mold_botrytis', 'healthy', 'leaf_spot',
    'not_a_plant', 'powdery_mildew', 'viral_disease',
]
PHYSIOLOGICAL_CLASSES = ['leaf_scorch_underwatering', 'nutrient_deficiency']
CLASSES = sorted(
    BASE_CLASSES + (PHYSIOLOGICAL_CLASSES if INCLUDE_PHYSIOLOGICAL_CLASSES else [])
)
# `_drop_low_green_content` floor for gray_mold_botrytis — necrotic
# Botrytis foliage is mostly brown, so the gate is set low to keep
# gray-on-brown leaf shots while still rejecting agar plates / fruit.
BOTRYTIS_GREEN_MIN_RATIO = 0.04
SPLIT_ROOTS = (VAL_INDOMAIN, VAL_PRIMARY, VAL_STUDIO, TEMP_CAL, CONFORMAL_CAL)

def ensure_layout():
    for c in CLASSES:
        (TRAIN_ROOT / c).mkdir(parents=True, exist_ok=True)
    for d in SPLIT_ROOTS:
        d.mkdir(parents=True, exist_ok=True)
ensure_layout()

# ── Eval helpers (used by every later phase) ────────────────────
NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# Scale to half the box's vCPUs, capped at 12 — heavy CPU augs
# (JpegRecompress, WhiteBalanceShift, DefocusOrMotion) bottleneck the
# A100 if we cap too low; more workers than vCPU/2 adds context-switch
# overhead without further throughput.
EVAL_WORKERS = min(12, max(2, (os.cpu_count() or 2) // 2))
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def iter_images(root: Path):
    for p in Path(root).rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            yield p

def count_images(root):
    return sum(1 for _ in iter_images(root)) if Path(root).exists() else 0

def _seed_worker(worker_id):
    s = (SEED + worker_id) % (2**32)
    np.random.seed(s); random.seed(s)

def _loader_kwargs(workers, **extra):
    """DataLoader kwargs. Caller `extra` overrides defaults so e.g.
    `persistent_workers=False` from a one-off cell takes effect — important
    after a cell interrupt leaves stale workers attached to a closed dataset.
    """
    kw = dict(num_workers=workers,
              pin_memory=(device == 'cuda'),
              worker_init_fn=_seed_worker)
    if workers > 0:
        kw['persistent_workers'] = True
        kw['prefetch_factor'] = 2
    kw.update(extra)
    return kw

import inspect as _inspect
_IMAGEFOLDER_ALLOW_EMPTY = (
    'allow_empty' in _inspect.signature(ImageFolder.__init__).parameters
)

def make_eval_loader(root, img_size, batch=128, workers=EVAL_WORKERS, class_to_idx=None):
    """Build a deterministic eval loader whose label space matches training.

    `ImageFolder` indexes classes by sorted directory listing — an eval
    slice missing one trained class would silently shift the label IDs.
    `class_to_idx` is the train-time map; we remap `target` so y_true is
    comparable to model `argmax` on every slice.
    """
    tx = T.Compose([T.Resize(int(img_size * 1.05)),
                    T.CenterCrop(img_size), T.ToTensor(), NORM])
    if_kwargs = {'allow_empty': True} if _IMAGEFOLDER_ALLOW_EMPTY else {}
    target_transform = None
    if class_to_idx is not None:
        tmp = ImageFolder(str(root), **if_kwargs)
        unknown = [c for c in tmp.classes if c not in class_to_idx]
        if unknown:
            raise RuntimeError(
                f'Eval folder {root} has classes outside training taxonomy: {unknown}.'
            )
        slice_to_train = {tmp.class_to_idx[c]: class_to_idx[c] for c in tmp.classes}
        target_transform = slice_to_train.__getitem__
    ds = ImageFolder(str(root), transform=tx, target_transform=target_transform,
                     **if_kwargs)
    if len(ds) == 0:
        raise RuntimeError(f'Eval folder is empty: {root}.  Run §2 first.')
    dl = DataLoader(ds, batch_size=batch, shuffle=False, **_loader_kwargs(workers))
    return ds, dl

@torch.inference_mode()
def evaluate(model, dl, class_names, device, name=''):
    """Disease macro-F1 + per-class recall + confusion matrix + healthy-bias."""
    model.eval()
    y_true, y_pred = [], []
    for x, y in dl:
        logits = model(x.to(device, non_blocking=True))
        y_pred.extend(logits.argmax(1).detach().cpu().tolist())
        y_true.extend(y.tolist())
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    disease_idx = [i for i, n in enumerate(class_names) if n not in ('healthy', 'not_a_plant')]
    f1 = float(f1_score(y_true, y_pred, labels=disease_idx, average='macro', zero_division=0))
    recall = recall_score(y_true, y_pred, labels=list(range(len(class_names))),
                          average=None, zero_division=0)
    h = class_names.index('healthy') if 'healthy' in class_names else None
    bias = float((y_pred == h).mean() / max((y_true == h).mean(), 1e-9)) if h is not None else 0.0
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names)))).tolist()
    print(f'[{name}] disease_macro_f1={f1:.3f}  healthy_bias={bias:.2f}  n={len(y_true)}')
    if 'val_realworld' in name or 'final' in name:
        for c, r in zip(class_names, recall):
            print(f'    recall[{c:25s}] = {r:.3f}')
    return {'name': name, 'disease_macro_f1': f1, 'healthy_bias': bias,
            'per_class_recall': {c: float(r) for c, r in zip(class_names, recall)},
            'confusion_matrix': cm}

# Persisted run log — every epoch appends a row, useful for post-mortem.
RUN_LOG = Path('models/run_log.jsonl')
RUN_LOG.parent.mkdir(parents=True, exist_ok=True)
def log_run(row: dict):
    row = {'ts': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), **row}
    with RUN_LOG.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')


## §2 — Data acquisition + group-aware splits

Public sources only. No manual upload. Every saved image gets a
sidecar JSON so the splitter can be group-aware (`(source, plant_id)`
goes together, never split across train and a held-out set).

| Source | Role |
|---|---|
| **PlantDoc** (HF) | real-world train + `val_realworld` seed |
| **PlantVillage** (HF) | capped train + `val_studio`. PlantVillage **never** enters `val_realworld`, `temp_cal` or `conformal_cal` (would break exchangeability) |
| **iNaturalist** | healthy houseplant leaves (Ficus / Monstera / …), Botrytis supplement (**`Green Leaves` annotation** off for the fungus class), Animalia for NAP, and free-text physiological queries (`leaf scorch`, `chlorosis`, …) for the v2.2 deferred classes |
| **COCO val2017** | `not_a_plant` images, plant-like categories filtered out |
| **Open Images V7** | `not_a_plant` indoor-scene supplement (Couch / Lamp / Vase / Bookcase / …) — closes the indoor-clutter gap behind production NAP failures |

Set `REBUILD_GENERATED_SPLITS = False` if you only want to top-up an
existing run.


In [ ]:
# §2 — Data acquisition + group-aware splits
import functools, threading
from io import BytesIO
from PIL import Image as PILImage

UA = {'User-Agent': 'PlantVision/2.2 (research; contact: local-notebook)'}
REBUILD_GENERATED_SPLITS = True

if REBUILD_GENERATED_SPLITS:
    for d in (TRAIN_ROOT, VAL_ROOT):
        if d.exists():
            shutil.rmtree(d)
    ensure_layout()

# Module-level Session: TCP keep-alive across the per-image fetches in
# this cell. Per-call ``headers=`` still wins over the session default.
_SESSION = requests.Session()
_SESSION.headers.update(UA)

_RETRY_STATUS = (429, 500, 502, 503, 504)

# Cross-call dedup of remote fetches (iNat photos can appear under
# multiple taxa). Guarded by a lock because we update it from worker
# threads in the ThreadPoolExecutor pools below.
_FETCH_SEEN_SHA: set = set()
_FETCH_SEEN_LOCK = threading.Lock()

# ── HTTP helpers (exponential backoff on 429/5xx) ────────────────
def http_get_json(url, params=None, timeout=30, retries=4):
    """GET + .json(). Logs every backoff so a starved class makes the
    upstream failure visible in the run log."""
    last, delay = None, 0.8
    for attempt in range(retries):
        try:
            r = _SESSION.get(url, params=params, timeout=timeout)
            if r.status_code in _RETRY_STATUS:
                last = requests.HTTPError(f'{r.status_code} {r.reason} on {url}')
                print(f'  http_get_json {url} attempt {attempt+1}/{retries} '
                      f'-> {r.status_code}; sleeping {delay:.1f}s')
                time.sleep(delay); delay *= 2; continue
            r.raise_for_status(); return r.json()
        except Exception as e:
            last = e
            print(f'  http_get_json {url} attempt {attempt+1}/{retries} '
                  f'-> {type(e).__name__}: {e}; sleeping {delay:.1f}s')
            time.sleep(delay); delay *= 2
    raise last if last is not None else RuntimeError(f'http_get_json exhausted retries: {url}')

def http_download(url, dest, timeout=120, retries=4):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + '.tmp')
    last, delay = None, 1.0
    for _ in range(retries):
        try:
            with _SESSION.get(url, stream=True, timeout=timeout) as r:
                if r.status_code in _RETRY_STATUS:
                    last = requests.HTTPError(f'{r.status_code} {r.reason} on {url}')
                    time.sleep(delay); delay *= 2; continue
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(1 << 20):
                        if chunk: f.write(chunk)
            tmp.replace(dest); return dest
        except Exception as e:
            last = e
            if tmp.exists():
                try: tmp.unlink()
                except Exception: pass
            time.sleep(delay); delay *= 2
    raise last if last is not None else RuntimeError(f'http_download exhausted retries: {url}')

def http_get_bytes(url, timeout=20, retries=4):
    """Returns body bytes or None — callers treat None as a soft skip
    so a single bad image doesn't abort the surrounding pool."""
    delay = 0.5
    for _ in range(retries):
        try:
            r = _SESSION.get(url, timeout=timeout)
            if r.status_code in _RETRY_STATUS:
                time.sleep(delay); delay *= 2; continue
            r.raise_for_status(); return r.content
        except Exception:
            time.sleep(delay); delay *= 2
    return None

def _stable_frac(text: str) -> float:
    return int(hashlib.sha1(text.encode('utf-8')).hexdigest()[:12], 16) / float(16**12 - 1)

def _save_rgb(img, out: Path, quality=92):
    out.parent.mkdir(parents=True, exist_ok=True)
    img.convert('RGB').save(out, quality=quality)

def _valid_image_bytes(body, min_side=160):
    try:
        im = PILImage.open(BytesIO(body)).convert('RGB')
        if min(im.size) < min_side: return None
        return im
    except Exception:
        return None

def _claim_sha(body: bytes) -> bool:
    """Try to claim a SHA-256. Returns True if this is the first time we've
    seen these bytes (and the caller may proceed), False on duplicate."""
    sha = hashlib.sha256(body).hexdigest()
    with _FETCH_SEEN_LOCK:
        if sha in _FETCH_SEEN_SHA:
            return False
        _FETCH_SEEN_SHA.add(sha)
    return True

def _fetch_save_remote(url, fp, ann_factory, *, timeout=20):
    """Fetch → validate as image → save RGB → write sidecar. Returns 1 on
    save, 0 on any skip (existing, failed fetch, bad image, dup)."""
    if fp.exists(): return 0
    body = http_get_bytes(url, timeout=timeout)
    if not body: return 0
    if not _claim_sha(body): return 0
    img = _valid_image_bytes(body)
    if img is None: return 0
    _save_rgb(img, fp)
    write_sidecar(fp, ann_factory())
    return 1

# ── iNaturalist (Leaves annotation, research-grade only) ────────
INAT_BASE = 'https://api.inaturalist.org/v1'
INAT_LEAF_TERM_ID, INAT_LEAF_TERM_VALUE_ID = 36, 38   # iNat 'Leaves' annotation, value='Green Leaves'

@functools.lru_cache(maxsize=64)
def inat_resolve_taxon(name, rank='genus'):
    res = http_get_json(f'{INAT_BASE}/taxa', {'q': name, 'rank': rank}).get('results') or []
    return res[0]['id'] if res else None

def inat_observations(taxon_id, max_n, leaf_only=True):
    rows, page, per = [], 1, 100
    while len(rows) < max_n:
        params = {'taxon_id': taxon_id, 'quality_grade': 'research', 'photos': 'true',
                  'per_page': per, 'page': page, 'order': 'desc', 'order_by': 'created_at'}
        if leaf_only:
            params['term_id'] = INAT_LEAF_TERM_ID
            params['term_value_id'] = INAT_LEAF_TERM_VALUE_ID
        batch = (http_get_json(f'{INAT_BASE}/observations', params).get('results') or [])
        if not batch: break
        rows.extend(batch)
        if len(batch) < per: break
        page += 1; time.sleep(0.25)
    return rows[:max_n]

def _photo_url(photo, size='medium'):
    raw = photo.get('url') or ''
    return re.sub(r'/square\.', f'/{size}.', raw) if raw.startswith('http') else None

def _save_one_inat(item, dest, klass, tx, label_confidence):
    obs, ph = item
    url = _photo_url(ph, 'medium')
    if not url: return 0
    fp = dest / f'inat_{tx}_{obs["id"]}_{ph["id"]}.jpg'
    if fp.exists(): return 0
    body = http_get_bytes(url)
    if not body: return 0
    if not _claim_sha(body): return 0
    img = _valid_image_bytes(body)
    if img is None: return 0
    _save_rgb(img, fp)
    write_sidecar(fp, CropAnnotation(
        plant_id=str(obs.get('id')),
        species=(obs.get('taxon') or {}).get('name'),
        diagnosis=klass, severity='unknown', leaf_or_whole_plant='leaf',
        capture_session=f'inat_taxon_{tx}', source='inat',
        label_confidence=label_confidence, bbox=[0, 0, 0, 0]))
    return 1

def download_inat_class(taxon_ids, dest, klass, max_per_taxon=160, leaf_only=True,
                        workers=8, label_confidence=0.7):
    dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for tx in taxon_ids:
        try:
            obs_list = inat_observations(tx, max_per_taxon, leaf_only=leaf_only)
        except Exception as e:
            print(f'  iNat taxon {tx} failed: {e}'); continue
        items = [(o, p) for o in obs_list for p in (o.get('photos') or [])[:1]]
        if not items: continue
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futs = [pool.submit(_save_one_inat, it, dest, klass, tx, label_confidence) for it in items]
            for f in as_completed(futs):
                n += f.result()
    return n

INAT_HEALTHY_TAXA = [
    ('Ficus','genus'),('Monstera','genus'),('Epipremnum','genus'),('Calathea','genus'),
    ('Spathiphyllum','genus'),('Dracaena','genus'),('Sansevieria','genus'),('Philodendron','genus'),
]
# The genus-level ``('Botrytis','genus')`` entry already pulls every
# species below it — a genus taxon_id query on iNat returns observations
# of all descendant species. The species-level rows are belt-and-braces
# only (they cost nothing in output: cross-call SHA dedup drops the
# re-fetched photos), kept solely so a future iNat genus re-org can't
# silently strand the host-specific species. Measured May 2026: with or
# without them the deduped foliage yield is the same ~138 images.
INAT_BOTRYTIS_TAXA = [
    ('Botrytis cinerea','species'),('Botrytis','genus'),
    ('Botryotinia','genus'),
    ('Botrytis pseudocinerea','species'),('Botrytis paeoniae','species'),
    ('Botrytis fabae','species'),('Botrytis tulipae','species'),
    ('Botrytis allii','species'),('Botrytis elliptica','species'),
    ('Botrytis squamosa','species'),
]

def _green_content_ratio(img) -> float:
    """Fraction of foliage-green pixels in HSV. Cheap heuristic for filtering
    fungal observations that aren't actually on a leaf (agar plates, dead
    tissue, fruit). Indoor leaf shots score 0.30–0.70; non-foliage close-ups
    score < 0.05."""
    import cv2 as _cv2
    arr = _cv2.cvtColor(np.asarray(img.convert('RGB')), _cv2.COLOR_RGB2HSV)
    h, s, v = arr[..., 0], arr[..., 1], arr[..., 2]
    mask = (h >= 35) & (h <= 85) & (s >= 40) & (v >= 50)
    return float(mask.mean())

def _green_ratio_for_path(p: Path):
    try:
        return p, _green_content_ratio(PILImage.open(p))
    except Exception:
        return p, None

def _drop_low_green_content(class_dir: Path, min_ratio: float = 0.10) -> int:
    """Remove images whose green-content ratio falls below the floor.
    Sidecars are removed alongside. PIL+cv2 work is GIL-released so we
    parallelise across EVAL_WORKERS threads."""
    paths = list(iter_images(class_dir))
    if not paths:
        return 0
    dropped = 0
    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        for p, ratio in pool.map(_green_ratio_for_path, paths):
            if ratio is None or ratio >= min_ratio:
                continue
            sc = p.with_suffix('.json')
            try:
                p.unlink()
                if sc.exists(): sc.unlink()
                dropped += 1
            except OSError:
                pass
    return dropped

def fetch_inat_supplement(healthy_per_taxon=180, botrytis_per_taxon=2400):
    healthy_ids = [t for t in (inat_resolve_taxon(n, r) for n, r in INAT_HEALTHY_TAXA) if t]
    n = download_inat_class(healthy_ids, TRAIN_ROOT / 'healthy', 'healthy',
                            max_per_taxon=healthy_per_taxon, leaf_only=True,
                            label_confidence=0.75)
    print(f'iNat healthy += {n}')
    # Botrytis is a fungus, not a plant — iNat's `Green Leaves` annotation
    # returns ~0 hits, so we fetch leaf_only=False and accept some non-leaf
    # hosts. The green-content gate filters obvious non-foliage. v2.2 raises
    # botrytis_per_taxon to compensate for Openverse being dropped.
    botrytis_ids = [t for t in (inat_resolve_taxon(n, r) for n, r in INAT_BOTRYTIS_TAXA) if t]
    n = download_inat_class(botrytis_ids, TRAIN_ROOT / 'gray_mold_botrytis', 'gray_mold_botrytis',
                            max_per_taxon=botrytis_per_taxon, leaf_only=False,
                            label_confidence=0.55)
    dropped = _drop_low_green_content(TRAIN_ROOT / 'gray_mold_botrytis', min_ratio=BOTRYTIS_GREEN_MIN_RATIO)
    print(f'iNat gray_mold_botrytis supplement += {n}, dropped {dropped} non-foliage')

# ── COCO val2017 → not_a_plant ──────────────────────────────────
COCO_CACHE = ROOT / 'cache_coco_val2017'
# Mirror used when images.cocodataset.org serves a CloudFront certificate
# whose SAN doesn't list the host (intermittent, currently broken).
# ``pcuenq/coco-2017-mirror`` exposes the canonical val2017.zip (816 MB)
# and annotations_trainval2017.zip (253 MB) as direct downloadable files
# under their canonical filenames — same SHA, same internal layout.
_COCO_HF_MIRROR = 'pcuenq/coco-2017-mirror'

def _coco_download(canonical_url: str, hf_filename: str, dest: Path) -> Path:
    """Fetch a COCO asset; fall back to ``shunk031/MSCOCO`` on SSL error."""
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    try:
        return http_download(canonical_url, dest)
    except requests.exceptions.SSLError as e:
        print(f'  COCO {canonical_url} SSL handshake failed ({e}); '
              f'falling back to {_COCO_HF_MIRROR} ({hf_filename})')
    from huggingface_hub import hf_hub_download
    src = hf_hub_download(_COCO_HF_MIRROR, hf_filename, repo_type='dataset')
    shutil.copy(src, dest)
    return dest

def fetch_coco_for_nap(target_n=1500):
    COCO_CACHE.mkdir(parents=True, exist_ok=True)
    img_zip, ann_zip = COCO_CACHE/'val2017.zip', COCO_CACHE/'annotations.zip'
    if not (COCO_CACHE/'val2017').exists():
        print('Downloading COCO val2017 (~1 GB)…')
        _coco_download('https://images.cocodataset.org/zips/val2017.zip',
                       'val2017.zip', img_zip)
        with zipfile.ZipFile(img_zip) as z: z.extractall(COCO_CACHE)
    if not (COCO_CACHE/'annotations').exists():
        _coco_download(
            'https://images.cocodataset.org/annotations/annotations_trainval2017.zip',
            'annotations_trainval2017.zip', ann_zip,
        )
        with zipfile.ZipFile(ann_zip) as z: z.extractall(COCO_CACHE)
    ann = json.loads((COCO_CACHE/'annotations'/'instances_val2017.json').read_text())
    # COCO is not exhaustively labelled, so dropping food / dining categories
    # cuts the worst plant-leak offenders without losing the indoor-clutter
    # frames we want for NAP.
    plant_like_names = {
        'potted plant', 'vase',
        'broccoli', 'carrot', 'banana', 'apple', 'orange',
        'sandwich', 'cake', 'donut', 'pizza', 'hot dog',
        'dining table', 'bowl',
    }
    plant_cat = {c['id'] for c in ann['categories'] if c['name'] in plant_like_names}
    plant_imgs = {a['image_id'] for a in ann['annotations'] if a['category_id'] in plant_cat}
    eligible = [im for im in ann['images'] if im['id'] not in plant_imgs]
    rng = random.Random(42); rng.shuffle(eligible)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for im in eligible[:target_n]:
        src, dst = COCO_CACHE/'val2017'/im['file_name'], dest/f'coco_{im["id"]}.jpg'
        if not src.exists() or dst.exists(): continue
        shutil.copy(src, dst)
        write_sidecar(dst, CropAnnotation(plant_id=f'coco_{im["id"]}', species=None,
            diagnosis='not_a_plant', severity='low', leaf_or_whole_plant='whole_plant',
            capture_session='coco_val2017', source='coco', label_confidence=0.99, bbox=[0,0,0,0]))
        n += 1
    print(f'COCO: not_a_plant += {n}')

# ── iNat Animalia → not_a_plant (indoor-clutter approximation) ─
# COCO is too studio-clean to teach the OOD gate about cats / dogs / birds
# in a leaf photo. iNat research-grade Animalia gets tagged
# source='inat_animalia' so the source-weighted sampler can distinguish
# pet-in-frame NAP from real-leaf iNat rows.
INAT_ANIMALIA_KINGDOM = 'Animalia'

def fetch_inat_animalia_for_nap(target_n=500):
    kingdom_id = inat_resolve_taxon(INAT_ANIMALIA_KINGDOM, rank='kingdom')
    if not kingdom_id:
        print('iNat Animalia kingdom not resolvable; skipping NAP supplement')
        return
    obs = inat_observations(kingdom_id, target_n, leaf_only=False)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    jobs = []
    for ob in obs:
        for ph in (ob.get('photos') or []):
            if len(jobs) >= target_n * 2: break  # oversample for skip slack
            url = _photo_url(ph, 'medium')
            if not url: continue
            fp = dest / f'inat_animalia_{ob["id"]}_{ph["id"]}.jpg'
            obs_id, taxon_name = ob['id'], (ob.get('taxon') or {}).get('name')
            ann = lambda obs_id=obs_id, taxon_name=taxon_name: CropAnnotation(
                plant_id=f'inat_animalia_{obs_id}', species=taxon_name,
                diagnosis='not_a_plant', severity='low',
                leaf_or_whole_plant='whole_plant',
                capture_session=f'inat_animalia_{kingdom_id}',
                source='inat_animalia', label_confidence=0.99,
                bbox=[0, 0, 0, 0])
            jobs.append((url, fp, ann))
        if len(jobs) >= target_n * 2: break
    n = 0
    pool = ThreadPoolExecutor(max_workers=8)
    try:
        futs = [pool.submit(_fetch_save_remote, u, f, a) for u, f, a in jobs]
        for fut in as_completed(futs):
            n += fut.result()
            if n >= target_n:
                break
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    print(f'iNat Animalia: not_a_plant += {n}')

# ── Open Images V7 → not_a_plant (indoor-clutter NAP) ─────────────
# v5 bbox CSV is the canonical validation-bbox file referenced by V6/V7
# (validation annotations didn't change between revisions).
OI_CACHE = ROOT / 'cache_openimages_v7'
OI_CDN_BASE = 'https://s3.amazonaws.com/open-images-dataset/validation/'
OI_VALIDATION_BBOX_URL = 'https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv'
OI_CLASS_CSV_URL = 'https://storage.googleapis.com/openimages/2018_04/class-descriptions-boxable.csv'

# MIDs resolved at runtime from class-descriptions-boxable.csv.
OI_INDOOR_LABELS = {
    'Couch', 'Vase', 'Lamp', 'Bookcase', 'Bed', 'Refrigerator',
    'Coffee table', 'Television', 'Pillow', 'Curtain', 'Toilet',
    'Sink', 'Furniture', 'Microwave oven', 'Oven', 'Sofa bed',
    'Chair', 'Desk', 'Cabinetry', 'Wardrobe', 'Nightstand',
}
OI_PLANT_LABELS = {
    'Houseplant', 'Plant', 'Flower', 'Tree', 'Leaf', 'Fruit',
    'Flowerpot',
}

def _oi_load_label_map():
    """LabelName MID → human-readable class name."""
    csv_local = OI_CACHE / 'class-descriptions-boxable.csv'
    if not csv_local.exists():
        OI_CACHE.mkdir(parents=True, exist_ok=True)
        http_download(OI_CLASS_CSV_URL, csv_local)
    mid_to_name = {}
    with csv_local.open() as fh:
        for line in fh:
            parts = line.rstrip('\n').split(',', 1)
            if len(parts) == 2:
                mid_to_name[parts[0]] = parts[1]
    return mid_to_name

def fetch_openimages_indoor(target_n=600):
    """NAP supplement: indoor-clutter images from Open Images V7 validation."""
    OI_CACHE.mkdir(parents=True, exist_ok=True)
    bbox_csv = OI_CACHE / 'validation-annotations-bbox.csv'
    if not bbox_csv.exists():
        print('Downloading Open Images V5 validation bbox CSV (~25 MB)…')
        http_download(OI_VALIDATION_BBOX_URL, bbox_csv)
    mid_to_name = _oi_load_label_map()
    indoor_mids = {m for m, n in mid_to_name.items() if n in OI_INDOOR_LABELS}
    plant_mids  = {m for m, n in mid_to_name.items() if n in OI_PLANT_LABELS}
    if not indoor_mids:
        print('Open Images: no indoor labels resolved; skipping NAP supplement')
        return
    per_image = {}  # image_id → set(label_mid)
    with bbox_csv.open() as fh:
        fh.readline()  # header
        for line in fh:
            parts = line.split(',', 3)
            if len(parts) < 3: continue
            per_image.setdefault(parts[0], set()).add(parts[2])
    eligible = [img for img, labs in per_image.items()
                if labs & indoor_mids and not (labs & plant_mids)]
    rng = random.Random(SEED + 11); rng.shuffle(eligible)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    # Oversample 2× so the pool has slack for misses (404s, decode failures,
    # files already present) and still hits target_n.
    jobs = []
    for img_id in eligible[:target_n * 2]:
        fp = dest / f'oi_{img_id}.jpg'
        ann = lambda img_id=img_id: CropAnnotation(
            plant_id=f'oi_{img_id}', species=None,
            diagnosis='not_a_plant', severity='low',
            leaf_or_whole_plant='whole_plant',
            capture_session='openimages_v7_validation',
            source='openimages_indoor', label_confidence=0.99,
            bbox=[0, 0, 0, 0])
        jobs.append((OI_CDN_BASE + img_id + '.jpg', fp, ann))
    n = 0
    pool = ThreadPoolExecutor(max_workers=8)
    try:
        futs = [pool.submit(_fetch_save_remote, u, f, a) for u, f, a in jobs]
        for fut in as_completed(futs):
            n += fut.result()
            if n >= target_n:
                break
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    print(f'Open Images indoor: not_a_plant += {n}')

# ── iNat → physiological classes ───────────────────────────────
# Free-text iNat queries for damage-related observations. Noisy (no curated
# annotation for "chlorosis" — iNat's Leaf annotations top out at
# `Coloured Leaves`). per_class is set high (400) because free-text recall
# is low and the round-robin claim below splits the deduped pool evenly
# across the two physiological classes. label_confidence is set low so
# KD + label-smoothing absorb the query noise.
INAT_PHYSIOLOGICAL_QUERIES = {
    'leaf_scorch_underwatering': [
        'leaf scorch', 'wilting leaves drought',
        'leaf curl drought', 'underwatered houseplant',
    ],
    'nutrient_deficiency': [
        'chlorosis leaf', 'iron chlorosis',
        'nitrogen deficiency leaf', 'magnesium deficiency leaf',
        'yellowing leaves plant', 'interveinal chlorosis',
        'potassium deficiency leaf', 'leaf yellowing nitrogen',
    ],
}

def _inat_freetext_observations(query, per_query):
    """Wrap the existing iNat layer for q= free-text search."""
    base = INAT_BASE + '/observations'
    params = {
        'q': query, 'quality_grade': 'research',
        'per_page': 100, 'photos': 'true', 'order': 'desc',
        'order_by': 'created_at',
    }
    page, out = 1, []
    while len(out) < per_query and page <= 5:
        params['page'] = page
        try:
            data = http_get_json(base, params=params)
        except Exception as e:
            print(f'  inat_freetext({query!r}) failed: {e}')
            break
        results = (data.get('results') or [])
        if not results: break
        out.extend(results)
        page += 1
    return out[:per_query]

def _save_inat_phys_observation(cls, q, ob, seen_obs) -> bool:
    """Try to save one iNat free-text observation under ``cls``. Returns
    True if a photo landed (and the obs is now claimed in ``seen_obs``),
    False otherwise. Failed photo fetches do NOT add to ``seen_obs`` so
    the sibling class can still try the same obs."""
    obs_id = ob['id']
    for ph in (ob.get('photos') or []):
        url = _photo_url(ph, 'medium')
        if not url:
            continue
        fp = TRAIN_ROOT / cls / f'inat_phys_{cls}_{obs_id}_{ph["id"]}.jpg'
        if fp.exists():
            seen_obs.add(obs_id)
            return True
        body = http_get_bytes(url)
        if not body or not _claim_sha(body):
            continue
        img = _valid_image_bytes(body)
        if img is None:
            continue
        _save_rgb(img, fp)
        write_sidecar(fp, CropAnnotation(
            plant_id=f'inat_phys_{obs_id}',
            species=(ob.get('taxon') or {}).get('name'),
            diagnosis=cls, severity='medium',
            leaf_or_whole_plant='leaf',
            capture_session=f'inat_phys_{q.replace(" ", "_")}',
            source='inat_physiological',
            label_confidence=0.55, bbox=[0, 0, 0, 0]))
        seen_obs.add(obs_id)
        return True
    return False

def fetch_inat_physiological(per_class=400):
    """Free-text iNat queries → physiological classes.

    Cross-class observation dedup is fairness-critical: a
    chlorosis-and-scorch photo can't land in both classes (the label
    signal would be contradictory). The two classes share several visual
    modes (yellowing leaves, generalised wilting), so first-come-first-
    served on the dict-insertion order silently privileges
    ``leaf_scorch_underwatering`` and starves ``nutrient_deficiency``
    of ~20% of its supply.

    Fix: fetch the candidate observation pools for every class up front,
    then claim observations in **round-robin** order. Each cycle every
    still-hungry class gets exactly one new observation (skipping any
    already claimed by a sibling), so both classes converge toward
    their target at the same rate. The green-content gate is
    intentionally NOT applied here — its HSV mask rejects pale-yellow
    chlorosis and brown leaf scorch as readily as it rejects agar
    plates.
    """
    targets: dict[str, int] = {}
    for cls in INAT_PHYSIOLOGICAL_QUERIES:
        dest = TRAIN_ROOT / cls
        dest.mkdir(parents=True, exist_ok=True)
        n_have = sum(1 for _ in iter_images(dest))
        if n_have >= per_class:
            print(f'iNat physiological ({cls}): already have {n_have}, skipping')
            continue
        targets[cls] = per_class - n_have
    if not targets:
        return

    pools: dict[str, list] = {}
    for cls, target in targets.items():
        queries = INAT_PHYSIOLOGICAL_QUERIES[cls]
        per_query = max(1, target // len(queries))
        pool = []
        for q in queries:
            pool.extend((q, ob) for ob in _inat_freetext_observations(q, per_query))
        pools[cls] = pool

    seen_obs: set = set()
    cursors = {cls: 0 for cls in targets}
    saved = {cls: 0 for cls in targets}
    done: set = set()
    while len(done) < len(targets):
        for cls in list(targets):
            if cls in done:
                continue
            if saved[cls] >= targets[cls]:
                done.add(cls); continue
            advanced = False
            while cursors[cls] < len(pools[cls]):
                q, ob = pools[cls][cursors[cls]]
                cursors[cls] += 1
                if ob['id'] in seen_obs:
                    continue
                if _save_inat_phys_observation(cls, q, ob, seen_obs):
                    saved[cls] += 1
                    advanced = True
                    break
            if not advanced:
                done.add(cls)

    for cls, n in saved.items():
        print(f'iNat physiological ({cls}): += {n}')

# ── PlantDoc (HF) → real-world train + val_realworld seed ───────
_PD_FUNGAL_KEYWORDS = (
    # Indoor-relevant leaf-spot morphologies. Rust, scab, and whole-plant
    # systemic patterns are intentionally OUT — pooling them with cercospora
    # / anthracnose / alternaria collapses recall on the actual indoor pattern.
    'leaf spot', 'mold', 'anthracnose', 'septoria', 'target spot',
    'early blight', 'late blight', 'leaf blight',
)
_PD_DROP_KEYWORDS = (
    'spider mite', 'mites',
    'rust',                     # very different morphology — drop
    'scab',                     # apple scab is more rust-like
    'black rot', 'fruit rot',   # fruit-stage disease, not leaf
)

def map_plantdoc_label(raw):
    s = re.sub(r'\s+', ' ', str(raw).strip().lower())
    if any(k in s for k in _PD_DROP_KEYWORDS):
        return None
    if 'bacterial spot' in s or 'bacterial blight' in s:
        return 'leaf_spot'
    if 'powdery mildew' in s:
        return 'powdery_mildew'
    if 'yellow virus' in s or 'mosaic virus' in s or 'curl virus' in s:
        return 'viral_disease'
    if any(k in s for k in _PD_FUNGAL_KEYWORDS):
        return 'leaf_spot'
    if 'leaf' in s:
        return 'healthy'
    return None

def _clip_bbox_xywh(bbox, w, h, pad=0.06):
    """Clip an XYWH (COCO-style) bbox to image bounds with padding.

    Strictly assumes XYWH (the HF card for `agyaatcoder/PlantDoc` confirms
    XYWH). Reject bboxes whose bottom-right corner falls outside the image:
    in valid XYWH `x + bw ≤ w` always holds, so an XYXY tuple sneaking
    through past the image centre will overflow this check.
    """
    x, y, bw, bh = [float(v) for v in bbox]
    if bw <= 1 or bh <= 1 or w <= 1 or h <= 1:
        return None
    # 5 % slack absorbs off-by-one annotations; anything past that is
    # either a real out-of-bounds bbox or XYXY misread as XYWH.
    if (x + bw) > w * 1.05 or (y + bh) > h * 1.05:
        return None
    if bw > w or bh > h:
        return None
    px, py = bw * pad, bh * pad
    x0, y0 = max(0, int(round(x - px))), max(0, int(round(y - py)))
    x1, y1 = min(w, int(round(x + bw + px))), min(h, int(round(y + bh + py)))
    if x1 - x0 < 32 or y1 - y0 < 32:
        return None
    return x0, y0, x1, y1

def _plantdoc_image_label(row, ds_split):
    """Resolve a PlantDoc image-level class label as a string, used as a
    fallback when the HF revision ships image-level labels instead of
    bbox-level ``objects``."""
    if 'label' in row:
        feat = ds_split.features.get('label') if hasattr(ds_split, 'features') else None
        if hasattr(feat, 'names') and feat.names:
            return feat.names[int(row['label'])]
        return row.get('label_name') or str(row['label'])
    return row.get('label_name') or row.get('class') or row.get('disease')

def fetch_plantdoc(max_train_per_class=500, max_val_per_class=120):
    try:
        ds = _hf_load_with_retry('agyaatcoder/PlantDoc')
    except Exception as e:
        print(f'PlantDoc HF download failed after retries: {e}')
        print('  HuggingFace 500/503 on /api/datasets/.../tree/... resolves in '
              'a few minutes — re-run this cell shortly.')
        return
    counts, dropped, no_obj_fallback = {}, 0, 0
    for split in [s for s in ('train', 'test') if s in ds]:
        out_base = VAL_PRIMARY if split == 'test' else TRAIN_ROOT
        cap = max_val_per_class if split == 'test' else max_train_per_class
        # HF returns rows clustered by class — shuffle so the cap samples
        # a wider mix.
        order = list(range(len(ds[split])))
        random.Random(SEED + (10 if split == 'test' else 11)).shuffle(order)
        for i in tqdm(order, desc=f'PlantDoc/{split}'):
            row = ds[split][i]
            img = row['image'].convert('RGB')
            w, h = img.size
            objects = row.get('objects') or {}
            bboxes = objects.get('bbox') or []
            cats = objects.get('category') or []
            # Object-level path (preferred — HF revisions ≥ 1.x).
            wrote_any = False
            for j, (bbox, cat) in enumerate(zip(bboxes, cats)):
                target = map_plantdoc_label(cat)
                if target is None or target not in CLASSES:
                    dropped += 1; continue
                key = (split, target)
                if counts.get(key, 0) >= cap:
                    continue
                box = _clip_bbox_xywh(bbox, w, h)
                if box is None: dropped += 1; continue
                crop = img.crop(box)
                image_id = row.get('image_id', i)
                stem = f'plantdoc_{split}_{image_id}_{j}'
                out = out_base / target / f'{stem}.jpg'
                if out.exists(): wrote_any = True; continue
                _save_rgb(crop, out)
                write_sidecar(out, CropAnnotation(
                    plant_id=f'plantdoc_{split}_{image_id}',
                    species=str(cat).split(' leaf')[0],
                    diagnosis=target, severity='unknown', leaf_or_whole_plant='leaf',
                    capture_session=f'plantdoc_{split}', source='plantdoc',
                    label_confidence=0.85, bbox=[int(v) for v in bbox]))
                counts[key] = counts.get(key, 0) + 1
                wrote_any = True
            if wrote_any:
                continue
            # Image-level fallback. Triggered when the HF revision drops
            # object-detection annotations and ships one category per image.
            raw = _plantdoc_image_label(row, ds[split])
            target = map_plantdoc_label(raw) if raw is not None else None
            if target is None or target not in CLASSES:
                dropped += 1; continue
            key = (split, target)
            if counts.get(key, 0) >= cap:
                continue
            image_id = row.get('image_id', i)
            stem = f'plantdoc_{split}_{image_id}_full'
            out = out_base / target / f'{stem}.jpg'
            if out.exists(): continue
            _save_rgb(img, out)
            write_sidecar(out, CropAnnotation(
                plant_id=f'plantdoc_{split}_{image_id}',
                species=str(raw).split(' leaf')[0] if raw else None,
                diagnosis=target, severity='unknown', leaf_or_whole_plant='whole_plant',
                capture_session=f'plantdoc_{split}', source='plantdoc',
                label_confidence=0.75, bbox=[0, 0, 0, 0]))
            counts[key] = counts.get(key, 0) + 1
            no_obj_fallback += 1
    print(f'PlantDoc counts: {counts}\nPlantDoc dropped: {dropped}'
          f'  image-level fallbacks: {no_obj_fallback}')

# ── PlantVillage (HF) → capped train + val_studio ───────────────
def _pv_normalise(s):
    return re.sub(r'[^a-z0-9_]+', '_', str(s).strip().lower()).strip('_')

PV_LABEL_MAP_RAW = {
    'Pepper,_bell___Bacterial_spot': 'leaf_spot',
    'Pepper__bell___Bacterial_spot': 'leaf_spot',
    'Tomato___Bacterial_spot':       'leaf_spot',
    'Peach___Bacterial_spot':        'leaf_spot',
    'Apple___healthy': 'healthy', 'Blueberry___healthy': 'healthy',
    'Cherry_(including_sour)___healthy': 'healthy', 'Corn_(maize)___healthy': 'healthy',
    'Grape___healthy': 'healthy',
    'Pepper,_bell___healthy': 'healthy', 'Pepper__bell___healthy': 'healthy',
    'Potato___healthy': 'healthy', 'Raspberry___healthy': 'healthy',
    'Soybean___healthy': 'healthy', 'Strawberry___healthy': 'healthy',
    'Tomato___healthy': 'healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'leaf_spot',
    'Tomato___Septoria_leaf_spot': 'leaf_spot',
    'Tomato___Target_Spot':        'leaf_spot',
    'Tomato___Leaf_Mold':          'leaf_spot',
    'Tomato___Early_blight':       'leaf_spot',
    'Tomato___Late_blight':        'leaf_spot',
    'Potato___Early_blight':       'leaf_spot',
    'Potato___Late_blight':        'leaf_spot',
    # Apple scab is rust-family morphology; keep OUT of leaf_spot.
    'Apple___Apple_scab':          None,
    # Apple black rot is a fruit disease — many images are whole-fruit.
    'Apple___Black_rot':           None,
    # Rust morphology is dropped wholesale.
    'Apple___Cedar_apple_rust':    None,
    'Corn_(maize)___Common_rust_': None,
    'Corn_(maize)___Northern_Leaf_Blight': 'leaf_spot',
    'Grape___Black_rot':           None,                  # fruit disease
    'Grape___Esca_(Black_Measles)':'leaf_spot',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 'leaf_spot',
    'Orange___Haunglongbing_(Citrus_greening)':   None,   # HLB ≠ leaf-spot
    'Cherry_(including_sour)___Powdery_mildew': 'powdery_mildew',
    'Squash___Powdery_mildew':                  'powdery_mildew',
    'Tomato___Tomato_mosaic_virus':           'viral_disease',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'viral_disease',
    'Tomato___Spider_mites Two-spotted_spider_mite': None,
    # Strawberry leaf scorch is physiological / abiotic, not fungal.
    'Strawberry___Leaf_scorch':  None,
    'Background_without_leaves': 'not_a_plant',
}
PV_LABEL_MAP = {_pv_normalise(k): v for k, v in PV_LABEL_MAP_RAW.items()}
# Each candidate is (hf_repo, hf_config_name_or_None, splits_tuple). The
# config name is required for mohanty/PlantVillage (ships three configs;
# load_dataset() refuses to pick one). BrandonFors has a single default
# config, so explicit name is None there.
PV_DATASET_CANDIDATES = [
    ('BrandonFors/Plant-Diseases-PlantVillage-Dataset', None,    ('train', 'test')),
    ('mohanty/PlantVillage',                            'color', ('train', 'test')),
]

def _hf_load_with_retry(name, config_name=None, *, split=None, retries=3, base_delay=5.0):
    """`datasets.load_dataset` with exponential-backoff retry on transient
    HuggingFace 5xx errors."""
    from datasets import load_dataset
    last = None
    delay = base_delay
    for attempt in range(retries):
        try:
            return load_dataset(name, config_name, split=split, token=False)
        except Exception as e:
            last = e
            msg = str(e)
            transient = (
                '500' in msg or '502' in msg or '503' in msg or '504' in msg
                or 'temporarily unavailable' in msg.lower()
                or 'connection' in msg.lower() or 'timeout' in msg.lower()
            )
            if attempt + 1 == retries or not transient:
                raise
            print(f'  HF {name} attempt {attempt + 1}/{retries} failed ({type(e).__name__}); '
                  f'retrying in {delay:.0f}s…')
            time.sleep(delay)
            delay *= 3
    if last is not None:
        raise last
    raise RuntimeError(f'_hf_load_with_retry exhausted retries for {name}')

def _load_first_available_hf_dataset(candidates):
    errors = []
    for entry in candidates:
        if len(entry) == 2:
            name, splits = entry; config_name = None
        else:
            name, config_name, splits = entry
        try:
            out = {}
            for split in [s for s in splits if s]:
                out[split] = _hf_load_with_retry(name, config_name, split=split)
            label = name + (f' [{config_name}]' if config_name else '')
            print(f'PlantVillage source: {label}')
            return name, out
        except Exception as e:
            errors.append(f'{name}: {e}')
    raise RuntimeError('All PlantVillage HF downloads failed:\n' + '\n'.join(errors))

def _row_label(row, ds):
    if 'label' in row:
        feat = ds.features.get('label') if hasattr(ds, 'features') else None
        if hasattr(feat, 'names') and feat.names:
            return feat.names[int(row['label'])]
        return row.get('label_name') or str(row['label'])
    return row.get('label_name') or row.get('class') or row.get('disease')

# PV `healthy` is uniform-bg / agar-plate; iNat `healthy` is real foliage.
# Cap PV-only contribution low so iNat dominates healthy.
PV_PER_CLASS_TRAIN_OVERRIDE = {'healthy': 250}

def fetch_plantvillage_into_train_and_studio(max_train_per_class=900, max_studio_per_class=300, studio_hash_frac=0.15):
    try:
        name, dsets = _load_first_available_hf_dataset(PV_DATASET_CANDIDATES)
    except Exception as e:
        print(f'PlantVillage HF download failed: {e}'); return
    # When the candidate ships a real `test` split, use IT as the canonical
    # val_studio source so train rows always go to TRAIN_ROOT.
    test_present = 'test' in dsets
    counts, dropped = {}, 0
    for split, ds in dsets.items():
        order = list(range(len(ds)))
        random.Random(SEED + (0 if split == 'train' else 1)).shuffle(order)
        for i in tqdm(order, desc=f'PlantVillage/{split}'):
            row = ds[i]
            raw = _row_label(row, ds)
            norm = _pv_normalise(raw)
            target = PV_LABEL_MAP.get(norm)
            if target is None or target not in CLASSES:
                dropped += 1; continue
            filename = row.get('image/filename') or row.get('filename') or f'{split}_{i}'
            stem = f'pv_{_pv_normalise(filename)}_{i}'
            if test_present:
                use_studio = (split == 'test')
            else:
                use_studio = (_stable_frac(stem) < studio_hash_frac)
            out_base = VAL_STUDIO if use_studio else TRAIN_ROOT
            if use_studio:
                cap = max_studio_per_class
            else:
                cap = PV_PER_CLASS_TRAIN_OVERRIDE.get(target, max_train_per_class)
            key = ('studio' if use_studio else 'train', target)
            if counts.get(key, 0) >= cap:
                continue
            out = out_base / target / f'{stem}.jpg'
            if out.exists(): continue
            img = row['image']
            if not hasattr(img, 'convert'):
                img = PILImage.open(img)
            _save_rgb(img.convert('RGB'), out)
            write_sidecar(out, CropAnnotation(
                plant_id=stem, species=str(raw).split('___')[0].replace('_', ' '),
                diagnosis=target, severity='unknown', leaf_or_whole_plant='leaf',
                capture_session=f'plantvillage_{split}', source='plantvillage',
                label_confidence=0.9, bbox=[0, 0, 0, 0]))
            counts[key] = counts.get(key, 0) + 1
    print(f'PV counts: {counts}\nPV dropped/ignored: {dropped}')

# ── Group-aware splits ──────────────────────────────────────────
def make_group_split(dest_root, ratio, seed, label, allowed_sources=None):
    """Move a fraction of TRAIN_ROOT into dest_root.

    Same `(source, plant_id)` lands together. `allowed_sources` constrains
    which sources can populate the split — used to keep PlantVillage out of
    the real-world / calibration splits.
    """
    rng = random.Random(seed); moved = 0
    for cls in CLASSES:
        cls_dir = TRAIN_ROOT / cls
        if not cls_dir.exists(): continue
        groups = {}
        for img in iter_images(cls_dir):
            sc = read_sidecar(img) or {}
            source = sc.get('source', 'unknown')
            if allowed_sources is not None and source not in allowed_sources:
                continue
            pid = sc.get('plant_id', img.stem)
            groups.setdefault((source, pid), []).append(img)
        ids = sorted(groups); rng.shuffle(ids)
        k = int(len(ids) * ratio)
        if ratio > 0 and len(ids) > 0:
            k = max(1, k)
        # Skip empty classes entirely — creating an empty subdir trips
        # torchvision.ImageFolder later.
        if k <= 0:
            continue
        out = dest_root / cls; out.mkdir(parents=True, exist_ok=True)
        for gid in ids[:k]:
            for img in groups[gid]:
                dst = out / img.name
                if dst.exists():
                    raise RuntimeError(f'Destination already exists: {dst}')
                shutil.move(str(img), str(dst))
                sc_path = img.with_suffix('.json')
                if sc_path.exists():
                    shutil.move(str(sc_path), str(dst.with_suffix('.json')))
                moved += 1
    print(f'{label} populated  (moved={moved})')

def _sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def _phash_file(p: Path):
    import imagehash
    try:
        return imagehash.phash(PILImage.open(p).convert('RGB'))
    except Exception:
        return None

def _dedup_train_in_place(root: Path, phash_threshold: int = 6) -> int:
    """Per-class SHA-256 + pHash dedup on TRAIN_ROOT before the splitters
    run. PIL.open + hashing both release the GIL, so we parallelise the
    per-file work and keep the per-class hash-set membership check serial."""
    drop = []
    for cls in CLASSES:
        cls_dir = Path(root) / cls
        if not cls_dir.exists():
            continue
        paths = sorted(iter_images(cls_dir))
        if not paths:
            continue
        with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
            shas   = list(pool.map(_sha256_file, paths))
            phashes = list(pool.map(_phash_file, paths))
        seen_sha, seen_ph = set(), []
        for p, sha, ph in zip(paths, shas, phashes):
            if sha in seen_sha:
                drop.append(p); continue
            seen_sha.add(sha)
            if ph is None:
                drop.append(p); continue
            if any(ph - existing <= phash_threshold for existing in seen_ph):
                drop.append(p); continue
            seen_ph.append(ph)
    for p in drop:
        sc = p.with_suffix('.json')
        try:
            p.unlink()
            if sc.exists(): sc.unlink()
        except OSError:
            pass
    print(f'pre-split dedup({root}): dropped={len(drop)}')
    return len(drop)

def make_val_realworld(ratio=0.15, seed=2):
    # Physiological-class sources join the realworld pool so the ship-gate
    # per-class recall check sees real rows. Every NAP source is included
    # so NAP recall is measured against indoor clutter, not just COCO.
    make_group_split(VAL_PRIMARY, ratio, seed, 'val_realworld',
                     allowed_sources={'inat', 'coco', 'plantdoc',
                                      'inat_physiological',
                                      'inat_animalia',
                                      'openimages_indoor'})
def make_val_indomain(ratio=0.10, seed=0):
    make_group_split(VAL_INDOMAIN, ratio, seed, 'val_indomain',
                     allowed_sources={'inat', 'coco', 'plantdoc',
                                      'inat_physiological',
                                      'inat_animalia',
                                      'openimages_indoor'})
def make_temperature_cal(ratio=0.12, seed=3):
    # All trained classes need calibration samples. Include every source
    # that feeds a trained class (NAP + physiological).
    make_group_split(TEMP_CAL, ratio, seed, 'temperature_cal',
                     allowed_sources={'plantdoc', 'inat',
                                      'coco', 'inat_animalia',
                                      'openimages_indoor',
                                      'inat_physiological'})
def make_conformal_cal(ratio=0.15, seed=1):
    make_group_split(CONFORMAL_CAL, ratio, seed, 'conformal_cal',
                     allowed_sources={'plantdoc', 'inat',
                                      'coco', 'inat_animalia',
                                      'openimages_indoor',
                                      'inat_physiological'})

# ── Run the pipeline ────────────────────────────────────────────
def _try_fetch(label, fn, *args, **kwargs):
    """Run a fetcher, log + continue on any exception. Upstream availability
    is the dominant failure mode for §2; a single transient failure after
    the slow fetches wastes ten minutes of work."""
    try:
        return fn(*args, **kwargs)
    except KeyboardInterrupt:
        raise
    except Exception as exc:  # noqa: BLE001 — by design; see above
        print(f'{label} skipped: {type(exc).__name__}: {exc}')
        log_run({'phase': 'fetch_skipped', 'label': label,
                 'error_type': type(exc).__name__, 'error': str(exc)})
        return None

_try_fetch('inat_supplement', fetch_inat_supplement,
           healthy_per_taxon=180, botrytis_per_taxon=2400)
_try_fetch('coco_nap', fetch_coco_for_nap, target_n=1500)
_try_fetch('inat_animalia_nap', fetch_inat_animalia_for_nap, target_n=500)
_try_fetch('openimages_indoor', fetch_openimages_indoor, target_n=600)
_try_fetch('plantdoc', fetch_plantdoc,
           max_train_per_class=500, max_val_per_class=120)
_try_fetch('plantvillage', fetch_plantvillage_into_train_and_studio,
           max_train_per_class=900, max_studio_per_class=300)
if INCLUDE_PHYSIOLOGICAL_CLASSES:
    _try_fetch('inat_physiological', fetch_inat_physiological, per_class=400)

# Dedup BEFORE splitting. The previous order — split first, dedup TRAIN_ROOT
# in §3 — left near-duplicates straddling train and held-out splits because
# make_group_split moves bytes by (source, plant_id) and only a SHA-256
# audit catches identical bytes (not pHash near-dupes).
_dedup_train_in_place(TRAIN_ROOT, phash_threshold=6)

make_val_realworld(); make_val_indomain(); make_temperature_cal(); make_conformal_cal()

def report_class_counts(root):
    print(f'\n{root}:')
    for c in CLASSES:
        print(f'  {c:25s} {count_images(root/c):5d}')
for r in (TRAIN_ROOT, VAL_PRIMARY, VAL_INDOMAIN, TEMP_CAL, CONFORMAL_CAL, VAL_STUDIO):
    report_class_counts(r)

# Per-class minimum training count. Most classes share the 150 default —
# the smallest count where KD distillation has produced trainable signal
# in practice for this taxonomy. ``gray_mold_botrytis`` is supplied by
# iNat alone (PlantDoc has no Botrytis label, PV's standard release has
# no Botrytis label) and ships chronically thin. Measured May 2026: the
# full Botrytis taxa set yields ~138 foliage images after the
# green-content gate, of which ~81 land in train after the four-way
# split drainage — that is the genuine iNat-only ceiling, not a
# starvation symptom. The 70 floor sits ~11 rows below it so a normal
# run passes while a real iNat outage (ConnectTimeout truncation,
# rate-limit wall) still fails loud. The §6 ship gate — disease
# macro-F1 ≥ 0.70 with no zero-recall class — is the real quality
# backstop for an under-supplied class. Openverse used to top this
# class up; removed in v2.2 because its anonymous tier was rate-capped
# below the notebook's burst rate.
THRESH_TRAIN_DEFAULT = 150
THRESH_TRAIN_PER_CLASS = {
    'gray_mold_botrytis': 70,
    # 60 is the floor at which KD + source-weighted sampling at 1.2x
    # produces trainable signal for thin iNat-free-text noised classes.
    'leaf_scorch_underwatering': 60,
    'nutrient_deficiency': 60,
}
THRESH_VAL = 5

def _train_floor_for(cls):
    return THRESH_TRAIN_PER_CLASS.get(cls, THRESH_TRAIN_DEFAULT)

under_train = [c for c in CLASSES if count_images(TRAIN_ROOT/c) < _train_floor_for(c)]
under_val   = [c for c in CLASSES if count_images(VAL_PRIMARY/c) < THRESH_VAL]
if under_train or under_val:
    msg = []
    if under_train:
        floors = {c: _train_floor_for(c) for c in under_train}
        msg.append(f'TRAIN_ROOT under per-class floor: {floors}')
    if under_val:   msg.append(f'VAL_PRIMARY < {THRESH_VAL}: {under_val}')
    raise RuntimeError('\n'.join(msg) +
        '\nTRAIN_ROOT < threshold means an upstream source for that class is '
        'starved. For gray_mold_botrytis the supply chain is iNat alone '
        '(PlantDoc has no Botrytis label); other classes pull from '
        'PlantDoc + PV + iNat. Check the per-source log lines above for '
        'download failures or rate-limit backoffs, then re-run §2 or '
        'drop additional real-world photos into TRAIN_ROOT/<class>/.')


## §3 — Dedup + Stage 1 detector + DINOv2 diagnostic probe

Three sanity passes before we burn GPU on the teacher:

1. **Dedup** the train set (per-class SHA-256 + pHash with Hamming ≤ 6).
   Cross-class similarities are kept on purpose — they are useful as
   hard negatives. Held-out splits are audited (SHA-256 only) but
   never pruned, because pruning a held-out set after the fact biases
   the gate.

2. **Stage 1 YOLOv8n** is re-trained from PlantDoc bboxes **by default**
   (`TRAIN_DETECTOR_FROM_SCRATCH = True`): `find_detector_yaml()` first
   looks for a user-supplied YOLO `data.yaml` under `data_v2/`, and when
   none exists `build_yolo_detection_dataset_from_plantdoc()` generates one
   from the same PlantDoc bboxes used for the classifier crops. Set
   `TRAIN_DETECTOR_FROM_SCRATCH = False` to skip Stage 1 and keep the
   shipped `detector.{pt,onnx}` pair instead.

3. **DINOv2-S linear probe** is the cheap "is the data even
   separable?" check. If linear-on-frozen-DINOv2 cannot get past 30 %
   on `val_realworld` after 8 epochs, the data is the problem, not
   the model.


In [ ]:
# §3 — Dedup + Stage 1 detector + DINOv2 probe
import imagehash
from PIL import Image as PILImage
from ultralytics import YOLO
import yaml

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

def dedup_directory(root, phash_threshold=6):
    """Per-class SHA-256 + pHash dedup. Per-file work runs in a thread pool
    (PIL + hashlib release the GIL); the per-class hash-set membership
    check stays serial."""
    drop = []
    for cls in CLASSES:
        cls_dir = Path(root) / cls
        if not cls_dir.exists(): continue
        paths = sorted(iter_images(cls_dir))
        if not paths:
            continue
        with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
            shas    = list(pool.map(sha256_file, paths))
            phashes = list(pool.map(_phash_file, paths))
        seen_sha, seen_phash = set(), []
        for p, s, ph in zip(paths, shas, phashes):
            if s in seen_sha:
                drop.append((p, 'sha256_dup')); continue
            seen_sha.add(s)
            if ph is None:
                drop.append((p, 'unreadable')); continue
            if any(ph - existing <= phash_threshold for existing in seen_phash):
                drop.append((p, 'phash_dup')); continue
            seen_phash.append(ph)
    for p, _reason in drop:
        sc = p.with_suffix('.json')
        try:
            p.unlink()
            if sc.exists(): sc.unlink()
        except Exception:
            pass
    print(f'dedup({root}): dropped={len(drop)}')

def assert_no_exact_split_leakage(train_root, heldout_roots):
    """SHA-256 cross-check between train and every held-out split."""
    train_paths = list(iter_images(train_root))
    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        train_shas = list(pool.map(sha256_file, train_paths))
    train_hash = dict(zip(train_shas, train_paths))
    problems = []
    for root in heldout_roots:
        if not Path(root).exists(): continue
        for p in iter_images(root):
            h = sha256_file(p)
            if h in train_hash:
                problems.append((str(train_hash[h]), str(p)))
                if len(problems) >= 10: break
        if len(problems) >= 10: break
    if problems:
        detail = '\n'.join(f'train={a}  heldout={b}' for a, b in problems)
        raise RuntimeError('Exact train/heldout leakage detected:\n' + detail)
    print('leakage audit (SHA): no exact overlap between train and held-out splits')

def resolve_phash_split_leakage(train_root, heldout_roots, threshold=6,
                                 max_drop_frac=0.25, max_report=10):
    """Perceptual-hash cross-check + auto-resolve. Source datasets (PlantDoc,
    PlantVillage) split near-duplicates across their published train/test
    boundaries; the §2 splitter can't stop those because they live on
    opposite sides of the HF split. This is the last-line defence before
    training reads the splits.

    Train data is never modified — held-out leaks are cheaper to lose
    than diverse training samples. A safety cap (``max_drop_frac``)
    raises if more than the given fraction of any held-out split would be
    removed.
    """
    # Group train pHashes by class. A held-out image is only checked
    # against train images of the **same class** so a leaf-spot photo
    # that pHash-collides with a (visually similar) bacterial photo
    # isn't dropped from val.
    train_hashes_by_class = {}
    train_paths = list(iter_images(train_root))
    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        train_phashes = list(pool.map(_phash_file, train_paths))
    for p, ph in zip(train_paths, train_phashes):
        if ph is not None:
            train_hashes_by_class.setdefault(p.parent.name, []).append(ph)
    if not any(train_hashes_by_class.values()):
        print('pHash leakage resolution: TRAIN_ROOT empty — skipped')
        return
    total_dropped, samples = 0, []
    for root in heldout_roots:
        if not Path(root).exists():
            continue
        held = list(iter_images(root))
        if not held:
            continue
        with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
            held_phashes = list(pool.map(_phash_file, held))
        to_drop = []
        for p, ph in zip(held, held_phashes):
            if ph is None:
                continue
            same_class_hashes = train_hashes_by_class.get(p.parent.name, [])
            best = min((ph - tph for tph in same_class_hashes), default=None)
            if best is not None and best <= threshold:
                to_drop.append((p, int(best)))
        frac = len(to_drop) / max(len(held), 1)
        if frac > max_drop_frac:
            preview = '\n'.join(f'  heldout={p}  pHash_dist={d}'
                                for p, d in to_drop[:max_report])
            raise RuntimeError(
                f'pHash leakage in {root}: would drop {len(to_drop)}/{len(held)} '
                f'({frac:.0%}) — exceeds max_drop_frac={max_drop_frac:.0%}. '
                f'Investigate the upstream split before re-running.\n' + preview
            )
        for p, _d in to_drop:
            sc = p.with_suffix('.json')
            try:
                p.unlink()
                if sc.exists(): sc.unlink()
            except OSError:
                pass
        if to_drop:
            print(f'  resolved {root}: dropped {len(to_drop)} of {len(held)} '
                  f'(near-duplicate of train, Hamming ≤ {threshold})')
            samples.extend((str(p), d) for p, d in to_drop[:max_report])
        total_dropped += len(to_drop)
    if total_dropped == 0:
        print(f'leakage audit (pHash): no near-duplicates within Hamming {threshold}')
    else:
        preview = '\n'.join(f'  {p}  pHash_dist={d}' for p, d in samples[:max_report])
        print(f'leakage resolution (pHash): dropped {total_dropped} held-out '
              f'near-duplicate(s) within Hamming {threshold}\n' + preview)

def dedup_each_split(heldout_roots, threshold=6):
    """Per-split internal dedup, in case the splitters moved a near-dup
    pair into the same held-out slice."""
    for root in heldout_roots:
        if Path(root).exists():
            dedup_directory(root, phash_threshold=threshold)

# §2 already ran _dedup_train_in_place(TRAIN_ROOT) before splitting; only
# dedup the held-out splits here.
dedup_each_split(SPLIT_ROOTS)
assert_no_exact_split_leakage(TRAIN_ROOT, SPLIT_ROOTS)
resolve_phash_split_leakage(TRAIN_ROOT, SPLIT_ROOTS, threshold=6)

# ── Stage 1 — YOLOv8n single-class plant_leaf ───────────────────
DETECTOR_DATA_ROOT = ROOT / 'detector'
TRAIN_DETECTOR_FROM_SCRATCH = True


def find_detector_yaml():
    """Look for a pre-existing YOLO ``data.yaml`` under ``data_v2/``.

    A user can drop their own labelled detection dataset here; when none
    exists we fall back to ``build_yolo_detection_dataset_from_plantdoc()``
    so Stage 1 can still train end-to-end from the same source we used
    for the classifier crops.
    """
    if not ROOT.exists():
        return None
    cands = [p for p in ROOT.rglob('data.yaml') if 'cache_coco' not in str(p)]
    return cands[0] if cands else None


def _yolo_label_ids(label_dir):
    ids = set()
    for txt in Path(label_dir).rglob('*.txt'):
        for line in txt.read_text().splitlines():
            parts = line.strip().split()
            if parts:
                try:
                    ids.add(int(float(parts[0])))
                except ValueError:
                    pass
    return ids


def build_yolo_detection_dataset_from_plantdoc(
    out_root: Path = DETECTOR_DATA_ROOT,
    train_ratio: float = 0.85,
    seed: int = 7,
) -> Optional[Path]:
    """Generate a YOLOv8 single-class detection dataset from PlantDoc.

    PlantDoc HF rows ship the full source image plus one or more
    ``{bbox: [x, y, w, h], category: <leaf-class>}`` objects. This helper
    walks the HF dataset (cached locally after the first §2 run), writes
    the full image to ``out_root/images/{train,val}/`` and a parallel
    YOLO ``.txt`` to ``out_root/labels/{train,val}/`` with one
    ``0 cx cy w h`` row per bbox — every category collapsed to a single
    class ``plant_leaf`` (id 0).

    PlantDoc's own ``test`` split lands entirely in YOLO ``val`` so the
    eval set matches the upstream hold-out. ``train`` rows are sub-split
    deterministically by ``seed`` at ``train_ratio``.

    Returns the generated ``data.yaml`` path, or ``None`` when the HF
    pull failed.
    """
    try:
        ds = _hf_load_with_retry('agyaatcoder/PlantDoc')
    except Exception as exc:
        print(f'PlantDoc HF pull for YOLO dataset failed: {exc}')
        return None

    if out_root.exists():
        shutil.rmtree(out_root)
    for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
        (out_root / sub).mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)
    train_count = val_count = skipped_no_bbox = 0
    for split_name in [s for s in ('train', 'test') if s in ds]:
        split = ds[split_name]
        order = list(range(len(split)))
        rng.shuffle(order)
        for i in tqdm(order, desc=f'YOLO/{split_name}'):
            row = split[i]
            objects = row.get('objects') or {}
            bboxes = objects.get('bbox') or []
            if not bboxes:
                skipped_no_bbox += 1
                continue
            img = row['image'].convert('RGB')
            w, h = img.size
            lines = []
            for bbox in bboxes:
                x, y, bw, bh = (float(v) for v in bbox)
                if bw <= 1 or bh <= 1:
                    continue
                cx = (x + bw / 2) / w
                cy = (y + bh / 2) / h
                nw = bw / w
                nh = bh / h
                if not (0 < cx < 1 and 0 < cy < 1 and 0 < nw <= 1 and 0 < nh <= 1):
                    continue
                lines.append(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
            if not lines:
                skipped_no_bbox += 1
                continue
            if split_name == 'test' or rng.random() > train_ratio:
                target = 'val'
            else:
                target = 'train'
            image_id = row.get('image_id', i)
            stem = f'plantdoc_{split_name}_{image_id}'
            img_path = out_root / 'images' / target / f'{stem}.jpg'
            lbl_path = out_root / 'labels' / target / f'{stem}.txt'
            if img_path.exists():
                continue
            _save_rgb(img, img_path)
            lbl_path.write_text('\n'.join(lines) + '\n')
            if target == 'train':
                train_count += 1
            else:
                val_count += 1

    if train_count == 0 or val_count == 0:
        print(f'PlantDoc → YOLO produced too few rows '
              f'(train={train_count} val={val_count}); skipping.')
        return None

    yaml_path = out_root / 'data.yaml'
    yaml_path.write_text(yaml.safe_dump({
        'path': str(out_root.resolve()),
        'train': 'images/train',
        'val': 'images/val',
        'names': ['plant_leaf'],
        'nc': 1,
    }, sort_keys=False))
    print(f'YOLO dataset built: train={train_count}  val={val_count}  '
          f'skipped_no_bbox={skipped_no_bbox}')
    print(f'  data.yaml = {yaml_path}')
    return yaml_path


yaml_path = find_detector_yaml()
if yaml_path is None and TRAIN_DETECTOR_FROM_SCRATCH:
    print('No YOLO data.yaml under data_v2/ — building one from PlantDoc bboxes.')
    yaml_path = build_yolo_detection_dataset_from_plantdoc()
if yaml_path is None:
    print(f'No YOLO dataset under {ROOT}.  Stage 1 training skipped — '
          f'the shipped detector.{{pt,onnx}} will be used at inference.')
else:
    spec = yaml.safe_load(yaml_path.read_text())
    for key in ('train', 'val'):
        p = spec.get(key)
        if isinstance(p, str):
            labels_dir = Path(yaml_path.parent, p).as_posix().replace('/images/', '/labels/')
            if Path(labels_dir).exists():
                ids = _yolo_label_ids(labels_dir)
                if any(i != 0 for i in ids):
                    raise RuntimeError(
                        f'{key} labels contain class ids {sorted(ids)}; remap to single class 0 first.'
                    )
    spec['names'], spec['nc'] = ['plant_leaf'], 1
    patched = yaml_path.parent / 'plant_leaf.yaml'
    patched.write_text(yaml.safe_dump(spec, sort_keys=False))
    detector = YOLO('yolov8n.pt')
    detector.train(
        # batch=128 saturates an A100 at imgsz=640 (~12 GB peak). YOLOv8
        # auto-falls back to a smaller batch on OOM so this stays safe on
        # smaller cards too.
        data=str(patched), epochs=60, patience=12, batch=128, imgsz=640,
        optimizer='AdamW', lr0=1e-3, cos_lr=True, amp=True,
        # Indoor leaf bboxes are tight — heavy mosaic = leaks across boxes.
        mosaic=0.1, hsv_h=0.0, hsv_s=0.4, hsv_v=0.4,
        device=0 if device == 'cuda' else 'cpu', workers=8,
        project='runs/detect', name='plant_leaf_indoor_v1', exist_ok=True,
    )
    best_pt = Path(detector.trainer.best) if getattr(detector, 'trainer', None) else next(
        Path('runs/detect').rglob('best.pt'))
    exported = YOLO(str(best_pt)).export(format='onnx', opset=13, dynamic=False)
    Path('models').mkdir(exist_ok=True)
    shutil.copy(best_pt, 'models/detector.pt')
    if exported and Path(exported).exists():
        shutil.copy(exported, 'models/detector.onnx')
    else:
        src = next(best_pt.parent.rglob('*.onnx'), None)
        if src: shutil.copy(src, 'models/detector.onnx')

# ── DINOv2-S linear probe (diagnostic) ──────────────────────────
# Backbone is frozen — extract features ONCE, then iterate the linear
# head over the cached embeddings. ~30× faster than recomputing the
# ViT-S forward pass per epoch on an A100.
PROBE_SIZE = 224
PROBE_TX = T.Compose([T.Resize(int(PROBE_SIZE * 1.05)),
                      T.CenterCrop(PROBE_SIZE), T.ToTensor(), NORM])
train_ds_probe = ImageFolder(str(TRAIN_ROOT), transform=PROBE_TX)
assert train_ds_probe.classes == sorted(CLASSES), (train_ds_probe.classes, sorted(CLASSES))
TRAIN_CLASS_TO_IDX = dict(train_ds_probe.class_to_idx)
train_dl_probe = DataLoader(
    train_ds_probe, batch_size=256, shuffle=False,
    **_loader_kwargs(EVAL_WORKERS, drop_last=False),
)
_, val_dl_probe = make_eval_loader(
    VAL_PRIMARY, PROBE_SIZE, batch=256, class_to_idx=TRAIN_CLASS_TO_IDX,
)

backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').eval().to(device)
for p in backbone.parameters(): p.requires_grad_(False)

@torch.inference_mode()
def _extract_features(dl):
    feats, labels = [], []
    for x, y in dl:
        f = backbone(x.to(device, non_blocking=True))
        feats.append(f.detach().cpu())
        labels.append(y.detach().cpu())
    return torch.cat(feats), torch.cat(labels)

print('caching DINOv2 features (train + val)…')
train_feats, train_labels = _extract_features(train_dl_probe)
val_feats,   val_labels   = _extract_features(val_dl_probe)
print(f'  train_feats={tuple(train_feats.shape)}  val_feats={tuple(val_feats.shape)}')

# Move to device for the head loop — small enough (<20 MB even at v2.2 scale).
train_feats_g = train_feats.to(device); train_labels_g = train_labels.to(device)
val_feats_g   = val_feats.to(device);   val_labels_g   = val_labels.to(device)

head = nn.Linear(train_feats.shape[1], len(train_ds_probe.classes)).to(device)
opt  = torch.optim.AdamW(head.parameters(), lr=3e-3, weight_decay=1e-4)
loss_fn_probe = nn.CrossEntropyLoss(label_smoothing=0.05)
PROBE_BATCH = 1024

probe_curve = []
g = torch.Generator(device='cpu').manual_seed(SEED)
for ep in range(8):
    head.train()
    perm = torch.randperm(train_feats_g.size(0), generator=g)
    for i in range(0, perm.size(0), PROBE_BATCH):
        idx = perm[i:i + PROBE_BATCH].to(device)
        opt.zero_grad(set_to_none=True)
        loss_fn_probe(head(train_feats_g[idx]), train_labels_g[idx]).backward()
        opt.step()
    head.eval()
    with torch.inference_mode():
        pred = head(val_feats_g).argmax(1)
        top1 = float((pred == val_labels_g).float().mean().item())
    probe_curve.append({'epoch': ep + 1, 'val_realworld_top1': top1})
    print(f'ep={ep+1}  val_realworld_top1={top1:.3f}')

Path('models/dinov2_probe.json').write_text(json.dumps(probe_curve, indent=2))
log_run({'phase': 'dinov2_probe', 'curve': probe_curve})
if probe_curve[-1]['val_realworld_top1'] < 0.30:
    print('WARN: DINOv2 frozen+probe < 0.30 — your data is hard. '
          'Audit labels in §2 before spending GPU on the teacher.')

# Per-class recall on val_realworld for the thin classes — surfaces a dead
# thin class here (cheap, on the frozen probe) instead of ~40 min later at
# the §6 ship gate. Warn-only by default; flip the flag to abort early.
PROBE_FAIL_FAST_ON_DEAD_THIN = False
_THIN_CLASSES = ('gray_mold_botrytis', 'leaf_scorch_underwatering', 'nutrient_deficiency')
with torch.inference_mode():
    _probe_pred = head(val_feats_g).argmax(1).cpu().numpy()
_probe_true = val_labels_g.cpu().numpy()
_dead_thin = []
print('thin-class probe recall (val_realworld):')
for _c in _THIN_CLASSES:
    _ci = TRAIN_CLASS_TO_IDX.get(_c)
    if _ci is None:
        continue
    _support = int((_probe_true == _ci).sum())
    if _support == 0:
        print(f'    recall[{_c:25s}] =   n/a  (no val_realworld samples)')
        continue
    _rec = float(((_probe_pred == _ci) & (_probe_true == _ci)).sum() / _support)
    print(f'    recall[{_c:25s}] = {_rec:.3f}  (support={_support})')
    if _rec == 0.0:
        _dead_thin.append(_c)
if _dead_thin:
    _msg = f'DINOv2 probe: zero recall on thin class(es) {_dead_thin} on val_realworld'
    if PROBE_FAIL_FAST_ON_DEAD_THIN:
        raise RuntimeError(_msg + ' — PROBE_FAIL_FAST_ON_DEAD_THIN=True. '
                           'Audit the source for these classes in §2 first.')
    print(f'WARN: {_msg} (warn-only; set PROBE_FAIL_FAST_ON_DEAD_THIN=True to abort).')

# Free memory before §4.
del backbone, head, train_dl_probe, val_dl_probe
del train_feats, train_feats_g, train_labels, train_labels_g
del val_feats,   val_feats_g,   val_labels,   val_labels_g
torch.cuda.empty_cache() if device == 'cuda' else None


## §4 — Loss, augs, teacher fine-tune (EfficientNetV2-S)

* **Loss:** `LogitAdjustmentLoss(τ=0.5, label_smoothing=0.05)`
  (Menon et al. ICLR 2021). Loss-side rebalancing matched to the
  **training prior** — no `WeightedRandomSampler` by default, because
  stacking the two over-corrects rare classes (Liu et al. 2024,
  *frontiersin*).
* **Sampler off by default.** Set `USE_WEIGHTED_SAMPLER = True` only
  if a single class dominates by > 10×.
* **Two-phase fine-tune:** head-only for 3 epochs (LR=3e-3), then
  unfreeze the last two backbone stages (head LR=5e-4, backbone
  LR=5e-5), early-stop on `val_indomain` disease macro-F1.
* **Augs:** indoor-targeted — WB shift, glare, defocus / motion blur,
  JPEG re-compression, mild ColorJitter, RandomErasing. **No** strong
  hue rotation / Solarize — colour is a disease signal.


In [ ]:
# §4 — Loss, augs, teacher fine-tune
import timm, torch.nn.functional as F
from collections import Counter
from contextlib import nullcontext
from torch.amp import autocast
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import WeightedRandomSampler
from PIL import ImageFilter, ImageEnhance, Image as PILImage
from io import BytesIO

USE_TORCH_COMPILE = True   # master switch; maybe_compile() warmup-guards it

USE_WEIGHTED_SAMPLER = False   # class-balanced sampler — off by default
# Per-image source weighting attacks the agricultural / houseplant domain
# mismatch. PlantDoc is field crops; the deployment target is indoor
# houseplants. Down-weighting PlantDoc rows in the sampler pulls the
# empirical training prior toward the real distribution without dropping
# the hard negatives KD leans on. The LA loss prior is recomputed from the
# sampler weights (``class_priors_from_source_weights``) so the correction
# stays aligned with the empirical per-batch distribution.
# Never enable alongside USE_WEIGHTED_SAMPLER (double rebalance).
USE_SOURCE_WEIGHTED_SAMPLER = True
SOURCE_WEIGHTS = {
    # Fix 1: rebalance toward the deployment/eval distribution. PlantDoc
    # (field photos) is what val_realworld is seeded from, so favour it;
    # PlantVillage studio images barely transfer; the NAP-only sources
    # (coco / inat_animalia / openimages_indoor) were over-weighted at 1.0.
    'plantdoc':            1.1,   # was 0.5 — favour the source the gate is graded on
    'plantvillage':        0.35,  # was 0.6 — studio images don't transfer to field eval
    'inat':                1.0,
    'inat_animalia':       0.5,   # was 1.0 — NAP-only, was over-weighted
    'inat_physiological':  1.2,
    'coco':                0.5,   # was 1.0 — NAP-only, was over-weighted
    'openimages_indoor':   0.5,   # was 1.0 — NAP-only, was over-weighted
}
LA_TAU = 0.5                   # logit adjustment τ — flattened post-cap
                                #   distribution justifies the lower value.
TAU_ABLATION = False
TAU_GRID = (0.3, 0.5, 0.7)
LABEL_SMOOTHING = 0.05

class LogitAdjustmentLoss(nn.Module):
    """Loss-side Logit Adjustment, Menon et al. ICLR 2021 eq. 4."""
    def __init__(self, class_priors, tau=LA_TAU, label_smoothing=LABEL_SMOOTHING):
        super().__init__()
        self.register_buffer('log_prior', torch.log(class_priors.clamp(min=1e-12)))
        self.tau, self.ls = tau, label_smoothing
    def forward(self, logits, target):
        return F.cross_entropy(logits + self.tau * self.log_prior, target,
                               label_smoothing=self.ls)

def class_priors_from_imagefolder(ds):
    counts = Counter([y for _, y in ds.samples])
    pri = torch.tensor([counts.get(i, 0) for i in range(len(ds.classes))], dtype=torch.float32)
    return pri.clamp(min=1.0) / pri.clamp(min=1.0).sum()

def class_priors_from_source_weights(ds, source_weights, default_weight=1.0):
    """Per-class prior under the source-weighted sampler.

    LogitAdjustment (Menon et al. ICLR 2021 eq. 4) assumes the ``log_prior``
    fed to the loss equals the per-batch class distribution the model
    actually sees during training. Under the source-weighted sampler the
    file-count prior is wrong; this helper sums each image's sampler weight
    into its class bucket, mirroring what WeightedRandomSampler applies at
    draw time.
    """
    counts = torch.zeros(len(ds.classes), dtype=torch.float32)
    for path, y in ds.samples:
        sc = read_sidecar(Path(path)) or {}
        src_name = sc.get('source')
        if not isinstance(src_name, str):
            src_name = 'unknown'
        w = max(float(source_weights.get(src_name, default_weight)), 0.0)
        counts[y] += w
    return counts.clamp(min=1e-6) / counts.clamp(min=1e-6).sum()

def make_class_balanced_sampler(ds, power=0.5):
    counts = Counter([y for _, y in ds.samples])
    weights = torch.DoubleTensor([(1.0 / counts[y]) ** power for _, y in ds.samples])
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True,
                                 generator=torch.Generator().manual_seed(SEED))

def make_source_weighted_sampler(ds, source_weights, default_weight=1.0):
    """Sample images at a rate proportional to their sidecar `source`. Images
    without a sidecar (or with an unknown source) get ``default_weight``."""
    drawn = Counter()
    sources_per_path = []
    weights = torch.zeros(len(ds.samples), dtype=torch.float64)
    for i, (path, _) in enumerate(ds.samples):
        sc = read_sidecar(Path(path)) or {}
        src_name = sc.get('source')
        if not isinstance(src_name, str):
            src_name = 'unknown'
        w = source_weights.get(src_name, default_weight)
        weights[i] = max(float(w), 0.0)
        drawn[src_name] += 1
        sources_per_path.append(src_name)
    total = weights.sum().item()
    if total <= 0:
        raise RuntimeError('source-weighted sampler: all weights are zero — '
                           'check sidecar `source` field and SOURCE_WEIGHTS')
    expected = {s: round(weights[[i for i, x in enumerate(sources_per_path) if x == s]].sum().item() / total, 4)
                for s in drawn}
    print(f'source-weighted sampler  raw_counts={dict(drawn)}  '
          f'expected_draw_fraction={expected}  weight_lookup={source_weights}')
    log_run({'phase': 'sampler', 'tag': 'source_weighted',
             'raw_counts': dict(drawn), 'expected_draw_fraction': expected,
             'weight_lookup': source_weights})
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True,
                                 generator=torch.Generator().manual_seed(SEED))

def amp_ctx():
    return autocast(device_type='cuda', dtype=torch.bfloat16) if device == 'cuda' else nullcontext()

class WhiteBalanceShift:
    def __init__(self, p=0.5, strength=(0.85, 1.15)):
        self.p, self.lo, self.hi = p, *strength
    def __call__(self, img):
        if random.random() > self.p: return img
        r_scale = random.uniform(self.lo, self.hi)
        b_scale = random.uniform(self.lo, self.hi)
        r, g, b = img.split()
        r = r.point(lambda v: max(0, min(255, int(v * r_scale))))
        b = b.point(lambda v: max(0, min(255, int(v * b_scale))))
        return PILImage.merge('RGB', (r, g, b))

class Glare:
    def __init__(self, p=0.25): self.p = p
    def __call__(self, img):
        return img if random.random() > self.p else \
               ImageEnhance.Brightness(img).enhance(random.uniform(1.05, 1.35))

class DefocusOrMotion:
    def __init__(self, p=0.3): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        f = (ImageFilter.GaussianBlur(radius=random.uniform(0.6, 1.8))
             if random.random() < 0.5
             else ImageFilter.BoxBlur(random.uniform(0.8, 2.0)))
        return img.filter(f)

class JpegRecompress:
    def __init__(self, p=0.35, q_range=(50, 92)): self.p, self.q_range = p, q_range
    def __call__(self, img):
        if random.random() > self.p: return img
        buf = BytesIO()
        img.save(buf, format='JPEG', quality=random.randint(*self.q_range))
        buf.seek(0)
        return PILImage.open(buf).convert('RGB').copy()

indoor_train_tx = T.Compose([
    T.Resize(int(256 * 1.15)),
    T.RandomResizedCrop(256, scale=(0.70, 1.0), ratio=(0.88, 1.14)),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.RandomPerspective(distortion_scale=0.12, p=1.0)], p=0.20),
    WhiteBalanceShift(p=0.5), Glare(p=0.25), DefocusOrMotion(p=0.3), JpegRecompress(p=0.35),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.0),
    T.ToTensor(), NORM, T.RandomErasing(p=0.12, scale=(0.02, 0.08)),
])

# ── Teacher: EfficientNetV2-S ──────────────────────────────────
BATCH, IMG = 256, 256      # A100-tuned.
train_ds = ImageFolder(str(TRAIN_ROOT), transform=indoor_train_tx)
if len(train_ds) == 0:
    raise RuntimeError('TRAIN_ROOT empty — re-run §2.')
CLASS_NAMES = list(train_ds.classes)
assert CLASS_NAMES == sorted(CLASSES), (CLASS_NAMES, sorted(CLASSES))
TRAIN_CLASS_TO_IDX = dict(train_ds.class_to_idx)
N_CLASSES = len(CLASS_NAMES)

if USE_SOURCE_WEIGHTED_SAMPLER and not USE_WEIGHTED_SAMPLER:
    priors = class_priors_from_source_weights(train_ds, SOURCE_WEIGHTS).to(device)
    print(f'LA prior (sampler-weighted): '
          f'{ {n: round(float(p), 4) for n, p in zip(train_ds.classes, priors.cpu().tolist())} }')
else:
    priors = class_priors_from_imagefolder(train_ds).to(device)
loss_fn = LogitAdjustmentLoss(priors, tau=LA_TAU, label_smoothing=LABEL_SMOOTHING)

if USE_SOURCE_WEIGHTED_SAMPLER and USE_WEIGHTED_SAMPLER:
    raise RuntimeError('USE_SOURCE_WEIGHTED_SAMPLER and USE_WEIGHTED_SAMPLER are '
                       'mutually exclusive — they would compose into a double '
                       'rebalance on top of the logit-adjustment loss prior.')
LOADER_WORKERS = min(12, EVAL_WORKERS)
if USE_SOURCE_WEIGHTED_SAMPLER:
    sampler = make_source_weighted_sampler(train_ds, SOURCE_WEIGHTS)
    train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                          **_loader_kwargs(LOADER_WORKERS, drop_last=True))
elif USE_WEIGHTED_SAMPLER:
    sampler = make_class_balanced_sampler(train_ds, power=0.5)
    train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                          **_loader_kwargs(LOADER_WORKERS, drop_last=True))
else:
    train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED),
                          **_loader_kwargs(LOADER_WORKERS, drop_last=True))

val_indomain_ds, val_indomain_dl = make_eval_loader(
    VAL_INDOMAIN, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
val_realworld_ds, val_realworld_dl = make_eval_loader(
    VAL_PRIMARY, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
temp_cal_ds, temp_cal_dl = make_eval_loader(
    TEMP_CAL, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
try:
    val_studio_ds, val_studio_dl = make_eval_loader(
        VAL_STUDIO, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
    )
except (RuntimeError, FileNotFoundError):
    # torchvision>=0.15 raises FileNotFoundError when no class subdirs
    # exist (e.g. PlantVillage download was skipped). Studio is a transfer
    # monitor — never blocking.
    print('val_studio not available (no class subdirs); skipping studio monitor.')
    val_studio_ds = val_studio_dl = None

teacher = timm.create_model(
    'tf_efficientnetv2_s.in21k_ft_in1k', pretrained=True, num_classes=N_CLASSES,
).to(device)
# torch.compile reduces dispatch overhead. Use the default mode (not
# reduce-overhead): the latter would CUDA-graph-capture a static-shape
# branch and recompile on every train↔eval batch-size swap.
def maybe_compile(model, tag, example_shape):
    """torch.compile guarded by a warmup forward+backward.

    torch.compile is lazy: it builds Triton/Inductor kernels on the FIRST
    forward, which otherwise happens deep inside the training loop — long
    after the try/except around torch.compile() itself, so a broken backend
    (e.g. an Inductor worker subprocess that cannot see the GPU) crashes
    mid-epoch with no recovery. Running one warmup step inside the guard
    surfaces a dead backend immediately and falls back to eager before any
    training GPU time is spent. USE_TORCH_COMPILE is the master opt-out.
    """
    if not (USE_TORCH_COMPILE and device == 'cuda' and hasattr(torch, 'compile')):
        return model
    try:
        compiled = torch.compile(model, mode='default')
        was_training = model.training
        compiled.train()
        dummy = torch.randn(*example_shape, device=device)
        with amp_ctx():
            compiled(dummy).float().sum().backward()
        compiled.zero_grad(set_to_none=True)
        compiled.train(was_training)
        print(f'{tag}: torch.compile enabled (default) — warmup OK')
        return compiled
    except Exception as exc:  # noqa: BLE001
        print(f'{tag}: torch.compile disabled after warmup failure '
              f'({type(exc).__name__}: {exc}); using eager')
        return model

teacher = maybe_compile(teacher, 'teacher', (2, 3, IMG, IMG))

# Studio is a non-blocking transfer monitor, but a sudden drift is worth
# a loud line. Warn when bias falls outside this range.
STUDIO_BIAS_BAND = (0.7, 1.5)

def _maybe_warn_studio_bias(metrics, tag):
    if metrics is None:
        return
    bias = float(metrics.get('healthy_bias', 1.0))
    lo, hi = STUDIO_BIAS_BAND
    if not lo <= bias <= hi:
        print(f'WARN: [{tag}] studio healthy_bias={bias:.2f} outside '
              f'[{lo:.2f}, {hi:.2f}]  (transfer monitor; non-blocking)')

def _eval_under_amp(model, dl, tag):
    with amp_ctx():
        return evaluate(model, dl, CLASS_NAMES, device, tag)

def step_eval(model, tag):
    """Per-epoch eval used for early-stopping decisions only.

    val_realworld is intentionally NOT computed per epoch: it is the ship
    gate, and looking at it every epoch leaks via human inspection of
    run_log.jsonl. The single post-training measurement happens in §6
    against the deployed INT8 ORT session.
    """
    out = {'indomain': _eval_under_amp(model, val_indomain_dl, f'{tag} val_indomain')}
    if val_studio_dl is not None:
        out['studio'] = _eval_under_amp(model, val_studio_dl, f'{tag} val_studio')
        _maybe_warn_studio_bias(out['studio'], tag)
    row = {'phase': 'teacher', 'tag': tag,
           'indomain_disease_f1': out['indomain']['disease_macro_f1'],
           'indomain_healthy_bias': out['indomain']['healthy_bias']}
    if 'studio' in out:
        row['studio_disease_f1'] = out['studio']['disease_macro_f1']
        row['studio_healthy_bias'] = out['studio']['healthy_bias']
    log_run(row)
    return out

def _make_sched(opt, n_epochs, n_steps_per_epoch, cosine=True):
    """Linear-warmup + cosine schedule. Set ``cosine=False`` for a flat
    post-warmup LR — useful for short head-only phases where the cosine
    would otherwise decay to ~0 before the LR has done useful work."""
    if n_steps_per_epoch <= 0:
        raise RuntimeError('Training loader has zero batches; lower BATCH or add data.')
    warmup = LinearLR(opt, start_factor=0.1, total_iters=n_steps_per_epoch)
    if not cosine:
        return warmup
    return SequentialLR(opt, schedulers=[
        warmup,
        CosineAnnealingLR(opt, T_max=max(1, (n_epochs - 1) * n_steps_per_epoch)),
    ], milestones=[n_steps_per_epoch])

def _train_phase(model, opt, n_epochs, scheduler, tag, early_stop=False, patience=4):
    best_f1, best_state, plateau = -1.0, None, 0
    for ep in range(n_epochs):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device)
            with amp_ctx():
                logits = model(x)
                loss = loss_fn(logits, y)
            if not torch.isfinite(loss):
                print(f'WARN: non-finite loss at {tag} ep{ep+1}; skipping batch'); continue
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], max_norm=5.0)
            opt.step(); scheduler.step()
        m = step_eval(model, f'{tag} ep{ep+1}')
        if early_stop:
            f1 = m['indomain']['disease_macro_f1']
            if f1 > best_f1:
                best_f1, plateau = f1, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                plateau += 1
                if plateau >= patience:
                    print(f'early stop @ ep{ep+1}  (best f1={best_f1:.3f})')
                    break
    if early_stop and best_state is not None:
        model.load_state_dict(best_state)
        print(f'restored best teacher weights  best_indomain_f1={best_f1:.3f}')

# Phase 1 — head-only. Flat post-warmup LR + early-stop best restore so
# phase 2 starts from the strongest head, not the last epoch.
# torch.compile wraps the model in an OptimizedModule whose
# named_parameters() prefixes every name with ``_orig_mod.``; iterate
# over the unwrapped module so `re.match(r'blocks\.(\d+)\.', n)` and the
# string-in-name checks below still see the original parameter names.
_teacher_eager = getattr(teacher, '_orig_mod', teacher)
for n, p in _teacher_eager.named_parameters():
    p.requires_grad_('classifier' in n)
opt = torch.optim.AdamW([p for p in _teacher_eager.parameters() if p.requires_grad],
                        lr=3e-3, weight_decay=1e-4)
_train_phase(teacher, opt, 3, _make_sched(opt, 3, len(train_dl), cosine=False),
             'head', early_stop=True, patience=1)

# Phase 2 — unfreeze from blocks.3 onward (Fix 4)
block_idxs = sorted({int(m.group(1)) for n in dict(_teacher_eager.named_parameters())
                     if (m := re.match(r'blocks\.(\d+)\.', n))})
if not block_idxs:
    raise RuntimeError('teacher has no `blocks.N.*` parameters — backbone topology changed.')
# Fix 4: unfreeze blocks.3+ (was last-two). The studio field-crop ->
# indoor houseplant domain gap needs more backbone adaptation capacity.
top_blocks = [f'blocks.{i}' for i in block_idxs if i >= 3] or [f'blocks.{i}' for i in block_idxs[-2:]]
unfreeze = top_blocks + ['classifier']
for p in _teacher_eager.parameters(): p.requires_grad_(False)
for n, p in _teacher_eager.named_parameters():
    if any(k in n for k in unfreeze): p.requires_grad_(True)
head_params = [p for n, p in _teacher_eager.named_parameters() if 'classifier' in n and p.requires_grad]
backbone_params = [p for n, p in _teacher_eager.named_parameters()
                   if any(k in n for k in top_blocks) and p.requires_grad]
if not head_params or not backbone_params:
    raise RuntimeError(f'empty optimiser group: head={len(head_params)} backbone={len(backbone_params)}')
print(f'Phase 2 unfreeze: {top_blocks} + classifier')
opt = torch.optim.AdamW([
    {'params': head_params,     'lr': 5e-4},
    {'params': backbone_params, 'lr': 5e-5},
], weight_decay=1e-4)
_train_phase(teacher, opt, 15, _make_sched(opt, 15, len(train_dl)), 'tune', early_stop=True)


## §5 — Knowledge distillation into MobileNetV3-Large

* **Loss:** `α · KD + (1-α) · CE_LA`, with α=0.5, τ=4.0. CE keeps the
  hard target signal; KD softens the teacher's confidence into a
  matching distribution. We use the same `LogitAdjustmentLoss` for the
  CE branch so the student inherits the teacher's prior correction.
* **EMA student** (adaptive decay; `EMA_DECAY = 1 - 1/max(KD_steps/4, 100)`)
  — small-data regime; the EMA copy is what we evaluate, save and
  export. Vanilla weights are noisy step-to-step; EMA smooths them
  out without retraining. A fixed 0.999 left ~half the random-init
  weights in the EMA after a ~720-step run; the adaptive form tracks
  KD_EPOCHS × len(train_dl) so the half-life stays useful when
  either changes.
* **Early-stop on `val_indomain` disease macro-F1**, patience=3.
* **Best EMA state restored** before saving.


In [ ]:
# §5 — KD into MobileNetV3-Large with EMA student
from copy import deepcopy
from torchvision import models

T_KD, ALPHA = 4.0, 0.5
KD_EPOCHS = 10

# EMA half-life ≈ KD_steps_total / 4. Floor at 100 keeps a sensible decay
# (~0.99) for short smoke runs. Bumping KD_EPOCHS or BATCH does not
# require a hand-tuned constant — the formula self-adjusts.
_KD_STEPS_TOTAL = KD_EPOCHS * len(train_dl)
EMA_DECAY = 1.0 - (1.0 / max(_KD_STEPS_TOTAL / 4, 100))
print(f'EMA_DECAY={EMA_DECAY:.4f}  KD_steps={_KD_STEPS_TOTAL}  '
      f'half_life≈{int(round(0.6931 / max(1e-9, 1 - EMA_DECAY)))} steps')

def _unwrap(m):
    """Return the underlying nn.Module (strips torch.compile's OptimizedModule
    wrapper if present). State-dicts saved from compiled modules carry an
    `_orig_mod.` key prefix that the loader in §6 can't consume."""
    return getattr(m, '_orig_mod', m)

student = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
student.classifier[3] = nn.Linear(student.classifier[3].in_features, N_CLASSES)
student = student.to(device)

# Snapshot the EMA copy BEFORE compile so its state_dict keys stay clean.
ema_student = deepcopy(student).eval()
for p in ema_student.parameters():
    p.requires_grad_(False)

# Compile only the trainable branch. EMA stays eager — its state_dict is
# the one we ship.
student = maybe_compile(student, 'student', (2, 3, IMG, IMG))

@torch.no_grad()
def ema_update(ema, online, decay=EMA_DECAY):
    """EMA both parameters AND floating-point buffers (BN running_mean /
    running_var). Integer buffers (``num_batches_tracked``) track the
    latest value verbatim. Iterates over the **unwrapped** modules so the
    online↔EMA parameter order matches whether or not the online branch
    is compiled."""
    online_u, ema_u = _unwrap(online), _unwrap(ema)
    for p_ema, p_on in zip(ema_u.parameters(), online_u.parameters()):
        p_ema.mul_(decay).add_(p_on.detach(), alpha=1.0 - decay)
    for b_ema, b_on in zip(ema_u.buffers(), online_u.buffers()):
        if b_on.dtype.is_floating_point:
            b_ema.mul_(decay).add_(b_on.detach(), alpha=1.0 - decay)
        else:
            b_ema.copy_(b_on)

teacher.eval()
opt = torch.optim.AdamW([
    {'params': _unwrap(student).classifier.parameters(), 'lr': 5e-4},
    {'params': [p for n, p in _unwrap(student).named_parameters() if 'classifier' not in n], 'lr': 5e-5},
], weight_decay=1e-4)
sched = _make_sched(opt, KD_EPOCHS, len(train_dl))

# Top-K candidate snapshots. FP32 val_indomain F1 is the in-loop selection
# signal; §6 re-evaluates each candidate at INT8 on val_realworld and picks
# the winner. K=4 gives the realworld arbiter a meaningfully different pool.
TOPK = 4
candidates = []  # list of (val_indomain_f1, epoch, ema_state_cpu)
plateau, PATIENCE = 0, 3

def _snapshot(m):
    """state_dict of the underlying eager module so §6 can reload it into
    a fresh, uncompiled MobileNetV3 without `_orig_mod.` prefix mismatch."""
    return {k: v.detach().cpu().clone() for k, v in _unwrap(m).state_dict().items()}

def _push_candidate(f1: float, ep: int, state: dict, k: int = TOPK):
    candidates.append((f1, ep, state))
    candidates.sort(key=lambda x: -x[0])
    while len(candidates) > k:
        candidates.pop()

def step_eval_kd(online, ema, ep):
    """Per-epoch KD eval — logs ONLINE and EMA F1 side by side so EMA
    misconfiguration surfaces at ep1 instead of ep2."""
    online.eval(); ema.eval()
    with amp_ctx():
        o_in = evaluate(online, val_indomain_dl, CLASS_NAMES, device,
                        f'kd-online ep{ep} val_indomain')
        e_in = evaluate(ema,    val_indomain_dl, CLASS_NAMES, device,
                        f'kd-ema    ep{ep} val_indomain')
        out = {'online': {'indomain': o_in}, 'ema': {'indomain': e_in}}
        e_studio = None
        if val_studio_dl is not None:
            e_studio = evaluate(ema, val_studio_dl, CLASS_NAMES, device,
                                f'kd-ema    ep{ep} val_studio')
            out['ema']['studio'] = e_studio
            _maybe_warn_studio_bias(e_studio, f'kd-ema ep{ep}')
    log_run({
        'phase': 'kd', 'epoch': ep,
        'online_indomain_disease_f1':   o_in['disease_macro_f1'],
        'online_indomain_healthy_bias': o_in['healthy_bias'],
        'ema_indomain_disease_f1':      e_in['disease_macro_f1'],
        'ema_indomain_healthy_bias':    e_in['healthy_bias'],
        'ema_studio_disease_f1':        (e_studio or {}).get('disease_macro_f1'),
        'ema_studio_healthy_bias':      (e_studio or {}).get('healthy_bias'),
    })
    return out

# KD loss uses the LA-shifted CE for the hard-target term but a vanilla
# KD term on raw teacher logits — the LA shift is a loss-side correction
# only, not a logit transform, so KD targets remain the teacher's
# probability distribution.
for ep in range(KD_EPOCHS):
    student.train()
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device)
        with amp_ctx():
            with torch.no_grad():
                t_logits = teacher(x)
            s_logits = student(x)
            # Upcast teacher logits to fp32 before softmax so the soft-target
            # distribution retains entropy precision under bf16 autocast.
            t_soft = F.softmax(t_logits.float() / T_KD, dim=1)
            kd = F.kl_div(
                F.log_softmax(s_logits / T_KD, dim=1),
                t_soft,
                reduction='batchmean',
            ) * (T_KD * T_KD)
            ce = loss_fn(s_logits, y)
            loss = ALPHA * kd + (1 - ALPHA) * ce
        if not torch.isfinite(loss):
            print(f'WARN: non-finite KD loss at ep{ep+1}; skipping batch'); continue
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=5.0)
        opt.step(); sched.step()
        ema_update(ema_student, student)

    metrics = step_eval_kd(student, ema_student, ep + 1)
    f1 = metrics['ema']['indomain']['disease_macro_f1']
    if not candidates or f1 > candidates[0][0]:
        plateau = 0
    else:
        plateau += 1
    _push_candidate(f1, ep + 1, _snapshot(ema_student))
    if plateau >= PATIENCE:
        best_f1_now = candidates[0][0]
        best_ep_now = candidates[0][1]
        print(f'KD early stop @ ep{ep+1}  (best ep={best_ep_now} f1={best_f1_now:.3f})')
        break

if not candidates:
    raise RuntimeError('KD finished with zero candidate snapshots — check the loop.')

best_f1, best_ep, best_ema_state = candidates[0]
# Reload the EMA-best into BOTH branches via the unwrapped modules so the
# saved checkpoint matches a fresh, eager MobileNetV3 head exactly.
_unwrap(ema_student).load_state_dict(best_ema_state)
_unwrap(student).load_state_dict(best_ema_state)

Path('models').mkdir(exist_ok=True)
torch.save(_unwrap(student).state_dict(), 'models/classifier.pth')

# Persist runner-up candidate(s) so §6 can re-evaluate at INT8 without
# re-running KD.
KD_CANDIDATES = [
    {'rank': i, 'val_indomain_f1': float(c[0]), 'epoch': int(c[1]), 'state': c[2]}
    for i, c in enumerate(candidates)
]
torch.save(
    [{'rank': c['rank'], 'val_indomain_f1': c['val_indomain_f1'],
      'epoch': c['epoch'], 'state_dict': c['state']} for c in KD_CANDIDATES],
    'models/classifier_candidates.pt',
)
log_run({'phase': 'kd_done', 'best_indomain_f1': best_f1, 'best_ep': best_ep,
         'topk_candidates': [{'epoch': c['epoch'], 'val_indomain_f1': c['val_indomain_f1']}
                              for c in KD_CANDIDATES]})
print(f'KD complete  top-{TOPK} candidates: '
      + ', '.join(f"ep{c['epoch']}/{c['val_indomain_f1']:.3f}" for c in KD_CANDIDATES))

# Free the teacher — ONNX export only needs the student.
del teacher
torch.cuda.empty_cache() if device == 'cuda' else None


## §6 — Production export: ONNX → static INT8 + temperature + conformal

### Export & quantisation order of operations

1. Export FP32 ONNX from the **EMA student weights** for each of the
   top-K KD candidates. CPU export keeps any future CUDA-only ops out
   of the graph; `quant_pre_process` (symbolic-shape inference) is
   **required** and run on every candidate.
2. Static INT8 calibration. The QDQ path uses Percentile (99.999)
   calibration, excludes HardSwish / HardSigmoid / Mul / Sigmoid /
   GlobalAveragePool output quant, and skips the first Conv, the last
   two Gemms (classifier head) and the SE-block inner Mul/Gemm chain
   via `nodes_to_exclude`. Calibration data comes from `temperature_cal`
   first, with deterministic backfill from `train_ds` when the split is
   thinner than 512 samples.
3. If any candidate exceeds `FAIL_DRIFT=0.05` on val_realworld after
   QDQ, retry once with `QuantFormat.QOperator`. There is **no**
   dynamic-weight-only fallback (`quantize_dynamic` is a no-op on the
   Conv-heavy MobileNetV3-Large graph).
4. Winner = candidate with the highest INT8 val_indomain F1
   (fp32_indomain F1 if every candidate collapsed). val_realworld is
   reserved for the ship gate.
5. Drift gate on val_realworld:
   * `drift ≤ 0.02` → ship INT8.
   * `0.02 < drift ≤ 0.05` → ship INT8 with WARN.
   * `drift > 0.05` → **honest FP32 fallback**: write the FP32 graph
     to `classifier_fp32.onnx`, delete `classifier_int8.onnx`,
     record `int8.shipped = False` in `model_card.json`. The deploy
     bundle script can no longer mistake an FP32 graph for an INT8
     one.
6. Fit temperature on the SHIPPED artefact's logits (INT8 if it
   passed the drift gate, FP32 otherwise — never fit T on a
   collapsed graph).
7. Fit conformal RAPS on `conformal_cal` softmaxed through the fitted
   T. Use ~80/20 fit/audit when `n_cal ≥ 200`; otherwise fit on the
   full split and report `audit_n = 0`.
8. Final eval on every val slice via the shipped ORT session.
9. **Ship gate** runs BEFORE any declarative artefact is persisted.
   `classifier.pth` and the canonical ONNX files have already been
   written above (they are the cheapest things to overwrite next
   run), but `classes.json` / `temperature.json` / `conformal.json` /
   `training_metrics.json` / `model_card.json` only land on disk
   after the gate passes — a failed run leaves the previous build's
   declarative bundle untouched.


In [ ]:
# §6 — ONNX FP32 → static INT8 + temperature + conformal RAPS + ship gate
import datetime
import onnx
import onnxruntime as ort
from onnxruntime.quantization import (
    quantize_static, CalibrationDataReader,
    QuantFormat, QuantType, CalibrationMethod)
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
except ImportError as e:
    raise RuntimeError(
        'onnxruntime.quantization.shape_inference.quant_pre_process is required. '
        'Upgrade onnxruntime to >=1.17 and re-run §1 install.'
    ) from e

Path('models').mkdir(exist_ok=True)

# Target calibration set size. Per-channel INT8 scales need ≥ 256 samples
# to stabilise; if TEMP_CAL alone is too thin, we draw the deficit from
# the held-out tail of train_ds (deterministic, never used for training
# in the same run because the calibration shuffler bypasses the sampler).
CALIB_TARGET_N = 512

class CalibrationReader(CalibrationDataReader):
    """Pulls examples from TEMP_CAL first, then back-fills from train_ds
    with a fixed seed so the calibration distribution stays consistent
    across runs."""

    def __init__(self, primary_ds, backup_ds, n_target, input_name='input'):
        rng = np.random.RandomState(SEED)
        n_primary = min(len(primary_ds), n_target)
        primary_idx = rng.permutation(len(primary_ds))[:n_primary]
        examples = [primary_ds[i][0].unsqueeze(0).numpy().astype(np.float32)
                    for i in primary_idx]
        deficit = n_target - n_primary
        if deficit > 0 and backup_ds is not None and len(backup_ds) > 0:
            backup_idx = rng.permutation(len(backup_ds))[:deficit]
            examples.extend(
                backup_ds[i][0].unsqueeze(0).numpy().astype(np.float32)
                for i in backup_idx
            )
            print(f'  INT8 calibration: {n_primary} from TEMP_CAL + '
                  f'{len(examples) - n_primary} from train_ds backfill '
                  f'(target={n_target})')
        elif n_primary < n_target:
            print(f'WARN: INT8 calibration thin: {n_primary} samples available, '
                  f'target {n_target}. Per-channel scales will be noisier.')
        self._iter = iter([(x,) for x in examples])
        self.input_name = input_name

    def get_next(self):
        nxt = next(self._iter, None)
        return None if nxt is None else {self.input_name: nxt[0]}

# Same eval transform as TEMP_CAL — we reuse this for the train_ds backup.
_CAL_TX = T.Compose([T.Resize(int(IMG * 1.05)),
                     T.CenterCrop(IMG), T.ToTensor(), NORM])
_train_cal_ds = ImageFolder(str(TRAIN_ROOT), transform=_CAL_TX)


def _ort_session(path):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(path, sess_options=so, providers=['CPUExecutionProvider'])

def _ort_logits(sess, x_np):
    in_name = sess.get_inputs()[0].name
    return sess.run(None, {in_name: x_np})[0]

def eval_onnx(path, dl):
    sess = _ort_session(path)
    y_true, y_pred = [], []
    for x, y in dl:
        logits = _ort_logits(sess, x.numpy().astype(np.float32))
        y_pred.extend(logits.argmax(1).tolist())
        y_true.extend(y.tolist())
    disease_idx = [i for i, n in enumerate(CLASS_NAMES) if n not in ('healthy', 'not_a_plant')]
    return float(f1_score(y_true, y_pred, labels=disease_idx, average='macro', zero_division=0))

# Static-quantisation options. MobileNetV3-Large's HardSwish + SE block stack
# is the canonical INT8 trap on naive QDQ. The mitigations:
#  * calibrate_method=CalibrationMethod.Percentile (default MinMax tanks
#    per-channel scales on activation outliers; passed as a quantize_static
#    parameter, with the percentile value via the CalibPercentile extra_option).
#  * Exclude HardSwish / HardSigmoid / Mul / Sigmoid / GlobalAveragePool output
#    quant — these are the SE-block bottleneck ops.
STATIC_QUANT_EXTRA_OPTIONS = {
    # Calibration METHOD is a quantize_static() parameter (calibrate_method=),
    # NOT an extra_options key — a 'CalibMethod' key here is silently ignored
    # and calibration falls back to MinMax. 'CalibPercentile' below only takes
    # effect once calibrate_method=CalibrationMethod.Percentile is passed.
    'CalibPercentile': 99.999,
    'OpTypesToExcludeOutputQuantization': [
        'HardSigmoid', 'HardSwish', 'Mul', 'Sigmoid', 'GlobalAveragePool',
    ],
    'ActivationSymmetric': False,
    'WeightSymmetric': True,
}

def _resolve_se_node_names(onnx_path):
    """Walk the graph and collect node names that should stay FP32: the
    first Conv (input adapter), the final classifier Gemm, and every node
    sitting inside a Squeeze-Excitation block (matched on the stable
    torchvision MobileNetV3 'features.*.block.*.<scale_block>' substring
    or any explicit `/se/` path).
    """
    model = onnx.load(onnx_path)
    nodes = list(model.graph.node)
    out = []
    # First Conv — keep the input adapter FP.
    for n in nodes:
        if n.op_type == 'Conv':
            out.append(n.name); break
    # Final classifier Gemm — the output head is two Linears; pin both.
    gemms = [n.name for n in nodes if n.op_type == 'Gemm']
    out.extend(gemms[-2:] if len(gemms) >= 2 else gemms)
    # SE-block inner nodes — torchvision MobileNetV3 emits scale_activation
    # / scale_layer named under each SE block; the `Mul` that applies the
    # gating mask + the two SE Convs / Gemms feeding into it are the
    # collapse points.
    for n in nodes:
        nm = (n.name or '').lower()
        if 'scale_activation' in nm or '/se/' in nm or '.se.' in nm:
            out.append(n.name)
    return sorted(set(n for n in out if n))

def _quantize_static(fp32_path, int8_path, *, calib_ds, calib_backup,
                     nodes_to_exclude, quant_format=QuantFormat.QDQ):
    quantize_static(
        fp32_path, int8_path,
        calibration_data_reader=CalibrationReader(
            calib_ds, calib_backup, CALIB_TARGET_N),
        quant_format=quant_format,
        calibrate_method=CalibrationMethod.Percentile,
        activation_type=QuantType.QUInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
        nodes_to_exclude=nodes_to_exclude,
        extra_options=STATIC_QUANT_EXTRA_OPTIONS,
    )

# Per-candidate static INT8 export, retried under QOperator format if QDQ
# busts the drift gate. Dynamic-weight-only is a no-op on conv-heavy nets
# (only Gemm/MatMul get quantised, MobileNetV3-L has 2 — so it just hides
# a failure under an indistinguishable second failure).
from torchvision import models as _tvm
def _build_blank_student():
    s = _tvm.mobilenet_v3_large(weights=None)
    s.classifier[3] = nn.Linear(s.classifier[3].in_features, N_CLASSES)
    return s.cpu().eval()

def _export_and_eval_int8_candidate(state_dict, label,
                                     fail_drift=0.05):
    s = _build_blank_student()
    s.load_state_dict(state_dict)
    raw_path  = f'models/classifier_raw_{label}.onnx'
    fp32_path = f'models/classifier_{label}.onnx'
    int8_path = f'models/classifier_int8_{label}.onnx'
    # Force the legacy TorchScript exporter. torch>=2.10 defaults to the
    # Dynamo path, which emits symbolic shape relationships that
    # onnxruntime.quantization.shape_inference can't fully resolve.
    torch.onnx.export(
        s, torch.randn(1, 3, IMG, IMG), raw_path,
        opset_version=18, input_names=['input'], output_names=['logits'],
        dynamic_axes={'input': {0: 'N'}, 'logits': {0: 'N'}},
        dynamo=False,
    )
    quant_pre_process(raw_path, fp32_path, skip_symbolic_shape=False)
    fp32_indomain_f1  = eval_onnx(fp32_path, val_indomain_dl)
    fp32_realworld_f1 = eval_onnx(fp32_path, val_realworld_dl)
    nodes_to_exclude = _resolve_se_node_names(fp32_path)
    quant_mode = 'static_qdq'
    _quantize_static(fp32_path, int8_path,
                     calib_ds=temp_cal_ds, calib_backup=_train_cal_ds,
                     nodes_to_exclude=nodes_to_exclude,
                     quant_format=QuantFormat.QDQ)
    int8_indomain_f1  = eval_onnx(int8_path, val_indomain_dl)
    int8_realworld_f1 = eval_onnx(int8_path, val_realworld_dl)
    if fp32_realworld_f1 - int8_realworld_f1 > fail_drift:
        print(f'  static QDQ drift {fp32_realworld_f1 - int8_realworld_f1:+.4f} > {fail_drift} '
              f'for {label} — retrying with QuantFormat.QOperator.')
        _quantize_static(fp32_path, int8_path,
                         calib_ds=temp_cal_ds, calib_backup=_train_cal_ds,
                         nodes_to_exclude=nodes_to_exclude,
                         quant_format=QuantFormat.QOperator)
        int8_indomain_f1  = eval_onnx(int8_path, val_indomain_dl)
        int8_realworld_f1 = eval_onnx(int8_path, val_realworld_dl)
        quant_mode = 'static_qop'
    return {
        'fp32_path': fp32_path, 'int8_path': int8_path,
        'fp32_indomain_f1':  fp32_indomain_f1,
        'fp32_realworld_f1': fp32_realworld_f1,
        'int8_indomain_f1':  int8_indomain_f1,
        'int8_realworld_f1': int8_realworld_f1,
        'quant_mode': quant_mode,
        'nodes_to_exclude': nodes_to_exclude,
    }

print(f'Exporting + quantising top-{len(KD_CANDIDATES)} candidates…')
candidate_results = []
for c in KD_CANDIDATES:
    r = _export_and_eval_int8_candidate(c['state'], label=f"rank{c['rank']}")
    r.update(epoch=c['epoch'], val_indomain_f1=c['val_indomain_f1'], rank=c['rank'])
    candidate_results.append(r)
    print(f"  ep{c['epoch']:>3d}  fp32_indomain={r['fp32_indomain_f1']:.4f}  "
          f"int8_indomain={r['int8_indomain_f1']:.4f}  "
          f"fp32_realworld={r['fp32_realworld_f1']:.4f}  "
          f"int8_realworld={r['int8_realworld_f1']:.4f}  "
          f"quant={r['quant_mode']}")

# Winner selection: sort by int8_indomain_f1 (val_realworld is reserved
# for the ship gate). If every candidate exceeded fail_drift the int8
# column is noise — fall back to fp32_indomain to avoid shipping the
# worst FP32 in the pool.
_QUANT_FAIL_DRIFT = 0.05
_all_int8_collapsed = all(
    (r['fp32_indomain_f1'] - r['int8_indomain_f1']) > _QUANT_FAIL_DRIFT
    for r in candidate_results
)
if _all_int8_collapsed:
    print(f'  every candidate exceeded fail_drift={_QUANT_FAIL_DRIFT} '
          f'— sorting by fp32_indomain_f1 instead.')
    candidate_results.sort(key=lambda r: -r['fp32_indomain_f1'])
else:
    candidate_results.sort(key=lambda r: -r['int8_indomain_f1'])
winner = candidate_results[0]
print(f"  ★ winner: ep{winner['epoch']}  "
      f"int8_indomain_f1={winner['int8_indomain_f1']:.4f}  "
      f"int8_realworld_f1={winner['int8_realworld_f1']:.4f}")

# Promote the winner's FP32 graph (canonical) and stage the candidate
# int8 graph next to it. The final ship/no-ship decision on the int8
# file happens below at the drift gate.
import shutil as _sh
fp32_onnx = 'models/classifier.onnx'
winner_int8_staged = 'models/classifier_int8_winner.onnx'
_sh.copyfile(winner['fp32_path'], fp32_onnx)
_sh.copyfile(winner['int8_path'], winner_int8_staged)
for r in candidate_results:
    for p in (r['fp32_path'], r['int8_path']):
        try:
            if Path(p).resolve() not in (Path(fp32_onnx).resolve(),
                                          Path(winner_int8_staged).resolve()):
                Path(p).unlink(missing_ok=True)
        except OSError:
            pass
    rawp = Path(f"models/classifier_raw_rank{r['rank']}.onnx")
    if rawp.exists():
        rawp.unlink()

# Stage the winner's eager-module weights as the source of truth on disk.
winner_state = next(c['state'] for c in KD_CANDIDATES if c['rank'] == winner['rank'])
torch.save(winner_state, 'models/classifier.pth')

# Conformal split — used only for RAPS quantile fitting downstream.
cal_dataset, cal_dl = make_eval_loader(
    CONFORMAL_CAL, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)

# Mandatory post-quantisation drift gate on val_realworld. On fail we ship
# FP32 honestly under classifier_fp32.onnx (NOT renamed as classifier_int8)
# so the deploy bundle can't mistake the artefact.
f1_fp32 = winner['fp32_realworld_f1']
f1_int8 = winner['int8_realworld_f1']
drift = f1_fp32 - f1_int8
print(f'val_realworld disease F1  fp32={f1_fp32:.4f}  int8={f1_int8:.4f}  drift={drift:+.4f}')
WARN_DRIFT, FAIL_DRIFT = 0.02, 0.05
canonical_int8 = Path('models/classifier_int8.onnx')
canonical_fp32_fallback = Path('models/classifier_fp32.onnx')
if drift > FAIL_DRIFT:
    print(f'WARN: INT8 drift {drift:+.4f} > FAIL_DRIFT={FAIL_DRIFT} on '
          f'quant_mode={winner.get("quant_mode", "unknown")!r}. '
          f'Shipping FP32 honestly as classifier_fp32.onnx; '
          f'classifier_int8.onnx will NOT be emitted.')
    canonical_int8.unlink(missing_ok=True)
    _sh.copyfile(fp32_onnx, canonical_fp32_fallback)
    int8_shipped, shipped_artifact = False, 'classifier_fp32.onnx (int8 export failed drift gate)'
    # Calibration session for downstream T / conformal is the FP32 graph.
    eval_sess_path = fp32_onnx
else:
    if drift > WARN_DRIFT:
        print(f'WARN: INT8 drift {drift:+.4f} > WARN_DRIFT={WARN_DRIFT}; shipping anyway.')
    _sh.copyfile(winner_int8_staged, canonical_int8)
    canonical_fp32_fallback.unlink(missing_ok=True)
    int8_shipped, shipped_artifact = True, 'classifier_int8.onnx'
    eval_sess_path = str(canonical_int8)
Path(winner_int8_staged).unlink(missing_ok=True)

eval_sess = _ort_session(eval_sess_path)

def _collect_logits(sess, dl):
    all_logits, all_y = [], []
    for x, y in dl:
        all_logits.append(_ort_logits(sess, x.numpy().astype(np.float32)))
        all_y.append(y.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)

# Temperature scaling on the shipped artefact's logits (INT8 if drift gate
# passed, FP32 otherwise — never fit T on a collapsed graph).
logits_temp_np, y_temp_np = _collect_logits(eval_sess, temp_cal_dl)
logits_temp = torch.from_numpy(logits_temp_np)
y_temp = torch.from_numpy(y_temp_np).long()

log_T = torch.tensor(0.0, requires_grad=True)
opt_T = torch.optim.LBFGS([log_T], lr=0.05, max_iter=50)
def closure():
    opt_T.zero_grad()
    T_now = log_T.exp().clamp(0.05, 10.0)
    loss = F.cross_entropy(logits_temp / T_now, y_temp)
    loss.backward(); return loss
opt_T.step(closure)
T_val = float(log_T.detach().exp().clamp(0.05, 10.0))
T_FIT_SOURCE = 'int8' if int8_shipped else 'fp32_fallback'
temperature_payload = {
    'temperature': T_val,
    'source': T_FIT_SOURCE,
    '_note': f'Fitted with LBFGS on temperature_cal NLL using {T_FIT_SOURCE} ONNX logits.',
    'fit_sources': ['plantdoc', 'inat', 'coco', 'inat_animalia', 'openimages_indoor'],
}
print(f'fitted T={T_val:.3f}  source={T_FIT_SOURCE}  '
      f'temp_cal MaxLogit mean={logits_temp.max(1).values.numpy().mean():.2f}')

# Conformal RAPS on conformal_cal probabilities (after temperature).
def _raps_score(p_sorted, lam, k_reg):
    cs = np.cumsum(p_sorted)
    return cs + lam * np.maximum(0.0, np.arange(1, len(p_sorted)+1, dtype=np.float64) - k_reg)

def _eval_raps_set(probs, labels, q_hat, lambda_reg, k_reg):
    n = probs.shape[0]
    if n == 0:
        return float('nan'), float('nan')
    covered, set_sizes = 0, []
    for i in range(n):
        order = np.argsort(-probs[i])
        s = _raps_score(probs[i, order], lambda_reg, k_reg)
        above = np.where(s >= q_hat)[0]
        size = max(int(above[0]) + 1 if above.size else len(s), 1)
        set_sizes.append(size)
        covered += int(labels[i] in order[:size])
    return covered / n, float(np.mean(set_sizes))

def calibrate_q_hat(cal_probs, cal_labels, *, alpha=0.1, lambda_reg=0.01, k_reg=1,
                    audit_probs=None, audit_labels=None):
    n = cal_probs.shape[0]
    if n == 0:
        raise RuntimeError('Empty conformal calibration split.')
    nonconf = np.empty(n)
    for i in range(n):
        order = np.argsort(-cal_probs[i])
        s = _raps_score(cal_probs[i, order], lambda_reg, k_reg)
        nonconf[i] = s[int(np.where(order == cal_labels[i])[0][0])]
    q = float(np.quantile(nonconf, min(1.0, np.ceil((n+1) * (1-alpha)) / n), method='higher'))
    in_cov, in_size = _eval_raps_set(cal_probs, cal_labels, q, lambda_reg, k_reg)
    out_cov = out_size = float('nan'); audit_n = 0
    if audit_probs is not None and audit_probs.shape[0] > 0:
        audit_n = int(audit_probs.shape[0])
        out_cov, out_size = _eval_raps_set(audit_probs, audit_labels, q, lambda_reg, k_reg)
    primary_cov  = out_cov  if audit_n > 0 else in_cov
    primary_size = out_size if audit_n > 0 else in_size
    return {'method': 'raps', 'alpha': alpha, 'q_hat': q,
            'lambda_reg': lambda_reg, 'k_reg': k_reg,
            'calibration_n': int(n),
            'empirical_coverage': primary_cov,
            'empirical_coverage_kind': 'out_of_sample' if audit_n > 0 else 'in_sample',
            'mean_set_size': primary_size,
            'audit_n': audit_n,
            'in_sample_coverage': in_cov,
            'in_sample_mean_set_size': in_size}

logits_cal_np, y_cal_np = _collect_logits(eval_sess, cal_dl)
probs_cal = torch.softmax(torch.from_numpy(logits_cal_np) / T_val, dim=1).numpy()
# Use ~80% of conformal_cal for q_hat fitting; skip the audit half entirely
# when the split is small (RAPS quantile needs the larger fit set).
n_cal = int(y_cal_np.shape[0])
order_cal = np.random.RandomState(SEED + 7).permutation(n_cal)
if n_cal >= 200:
    fit_n = int(n_cal * 0.80)
    fit_idx, audit_idx = order_cal[:fit_n], order_cal[fit_n:]
    cfg = calibrate_q_hat(
        probs_cal[fit_idx], y_cal_np[fit_idx], alpha=0.10,
        audit_probs=probs_cal[audit_idx], audit_labels=y_cal_np[audit_idx],
    )
else:
    print(f'conformal: n_cal={n_cal} < 200 — fitting on the full split with audit_n=0')
    cfg = calibrate_q_hat(probs_cal, y_cal_np, alpha=0.10)
cfg['coverage_scope'] = ('all 9 classes (plantdoc, inat, coco, '
                          'inat_animalia, openimages_indoor); '
                          'not_a_plant is additionally guarded by the '
                          'Stage 1 detector + MaxLogit OOD floor.')
cfg['cal_sources'] = ['plantdoc', 'inat', 'coco', 'inat_animalia',
                       'openimages_indoor']
cfg['fit_source'] = T_FIT_SOURCE
print('conformal:', cfg)

# Calibrate the MaxLogit OOD floor on the shipped session. Pick the
# threshold at the TARGET_PLANT_FPR plant-MaxLogit percentile — at most
# 5% of plants get flagged by MaxLogit alone (the production OOD gate
# also requires low YOLO detector confidence, so the realised plant→NAP
# false-positive rate is much lower).
TARGET_PLANT_FPR = 0.05
NAP_CLASS_IDX = (CLASS_NAMES.index('not_a_plant')
                 if 'not_a_plant' in CLASS_NAMES else None)

def _maxlogit_distribution(sess, dl, *, only_class=None, exclude_class=None):
    out = []
    for x, y in dl:
        logits = _ort_logits(sess, x.numpy().astype(np.float32))
        for i in range(logits.shape[0]):
            yi = int(y[i])
            if only_class is not None and yi != only_class:
                continue
            if exclude_class is not None and yi == exclude_class:
                continue
            out.append(float(logits[i].max()))
    return np.array(out, dtype=np.float64)

plant_ml = _maxlogit_distribution(eval_sess, val_indomain_dl, exclude_class=NAP_CLASS_IDX)
nap_ml = (_maxlogit_distribution(eval_sess, val_realworld_dl, only_class=NAP_CLASS_IDX)
          if NAP_CLASS_IDX is not None else np.array([]))

if len(plant_ml) >= 30:
    ML_FLOOR = float(np.percentile(plant_ml, TARGET_PLANT_FPR * 100))
else:
    print(f'WARN: only {len(plant_ml)} plant MaxLogit samples — '
          f'falling back to 4.0 default')
    ML_FLOOR = 4.0

# Disable the gate when separation is the wrong sign — a positive floor
# would over-fire on plants while sparing NAPs.
ML_SEPARATION = (float(np.percentile(plant_ml, 50) - np.percentile(nap_ml, 50))
                  if len(plant_ml) and len(nap_ml) else None)
if ML_SEPARATION is not None and ML_SEPARATION < 0:
    print(f'WARN: NAP MaxLogit median above plant median (sep={ML_SEPARATION:+.3f}) '
          f'— disabling the MaxLogit gate (max_logit_floor=null).')
    ML_FLOOR_PERSISTED = None
else:
    ML_FLOOR_PERSISTED = ML_FLOOR

ML_CAL_META = {
    'target_plant_fpr': TARGET_PLANT_FPR,
    'plant_n': int(len(plant_ml)),
    'plant_p05': float(np.percentile(plant_ml, 5)) if len(plant_ml) else None,
    'plant_p50': float(np.percentile(plant_ml, 50)) if len(plant_ml) else None,
    'plant_p95': float(np.percentile(plant_ml, 95)) if len(plant_ml) else None,
    'nap_n': int(len(nap_ml)),
    'nap_p05': float(np.percentile(nap_ml, 5)) if len(nap_ml) else None,
    'nap_p50': float(np.percentile(nap_ml, 50)) if len(nap_ml) else None,
    'nap_p95': float(np.percentile(nap_ml, 95)) if len(nap_ml) else None,
    'separation_p50': ML_SEPARATION,
    'fit_source': T_FIT_SOURCE,
}
print(f'MaxLogit OOD floor calibrated: '
      f'{"off" if ML_FLOOR_PERSISTED is None else f"{ML_FLOOR_PERSISTED:.3f}"}  '
      f'(plant n={len(plant_ml)} p50={ML_CAL_META["plant_p50"]}, '
      f'nap n={len(nap_ml)} p50={ML_CAL_META["nap_p50"]})')

# Final metrics across all val slices via the shipped ORT session.
class _OnnxModelAdapter:
    """Thin wrapper so evaluate() can call model(x) on an ORT session."""
    def __init__(self, sess): self.sess = sess
    def eval(self): return self
    def __call__(self, x):
        out = _ort_logits(self.sess, x.detach().cpu().numpy().astype(np.float32))
        return torch.from_numpy(out).to(x.device)

eval_adapter = _OnnxModelAdapter(eval_sess)
ONNX_EVAL_DEVICE = 'cpu'
final_metrics = {
    'indomain':  evaluate(eval_adapter, val_indomain_dl,  CLASS_NAMES, ONNX_EVAL_DEVICE, 'final val_indomain'),
    'realworld': evaluate(eval_adapter, val_realworld_dl, CLASS_NAMES, ONNX_EVAL_DEVICE, 'final val_realworld'),
}
if val_studio_dl is not None:
    final_metrics['studio'] = evaluate(eval_adapter, val_studio_dl, CLASS_NAMES, ONNX_EVAL_DEVICE, 'final val_studio')

# Compose payloads (NOT yet written — the ship gate runs first).
classes_payload = {
    'model_version': MODEL_VERSION,
    'class_names': CLASS_NAMES,
    'detector_classes': ['plant_leaf'],
    'training': {
        'img_size': IMG,
        'batch_size': BATCH,
        'loss': 'logit_adjustment_ce',
        'logit_adjustment_tau': LA_TAU,
        'label_smoothing': LABEL_SMOOTHING,
        'use_weighted_sampler': bool(USE_WEIGHTED_SAMPLER),
        'use_source_weighted_sampler': bool(USE_SOURCE_WEIGHTED_SAMPLER),
        'source_weights': dict(SOURCE_WEIGHTS) if USE_SOURCE_WEIGHTED_SAMPLER else None,
        'optimizer': 'AdamW',
        'weight_decay': 1e-4,
        'lr_schedule': 'cosine + warmup',
        'precision': 'bf16-mixed' if device == 'cuda' else 'fp32',
        'primary_metric': 'val_realworld_disease_macro_f1',
        'primary_val_set': 'data_v2/classification/val/realworld',
        'isolated_val_slice': 'data_v2/classification/val/plantvillage_studio',
    },
    'distillation': {
        'teacher': 'tf_efficientnetv2_s.in21k_ft_in1k',
        'student': 'mobilenet_v3_large',
        'loss': 'alpha*KD + (1-alpha)*CE_LA',
        'alpha': ALPHA,
        'tau':   T_KD,
        'ema_decay': EMA_DECAY,
        'epochs': KD_EPOCHS,
    },
    'ood': {
        'score': 'max_logit',
        'max_logit_floor': ML_FLOOR_PERSISTED,
        '_score_note': 'MaxLogit (Hendrycks 2019) — robust under temperature scaling and label smoothing where MSP fails.',
        '_calibration': ML_CAL_META,
    },
}

model_card = {
    'version': MODEL_VERSION,
    'trained_at': datetime.datetime.now(datetime.timezone.utc).replace(microsecond=0).isoformat(),
    'classes': CLASS_NAMES,
    'img_size': IMG,
    'temperature': T_val,
    'conformal': cfg,
    'train_counts':           {c: count_images(TRAIN_ROOT  / c) for c in CLASS_NAMES},
    'val_realworld_counts':   {c: count_images(VAL_PRIMARY / c) for c in CLASS_NAMES},
    'val_indomain_counts':    {c: count_images(VAL_INDOMAIN/ c) for c in CLASS_NAMES},
    'temperature_cal_counts': {c: count_images(TEMP_CAL    / c) for c in CLASS_NAMES},
    'conformal_cal_counts':   {c: count_images(CONFORMAL_CAL/ c) for c in CLASS_NAMES},
    'val_studio_counts':      {c: count_images(VAL_STUDIO  / c) for c in CLASS_NAMES} if val_studio_dl is not None else None,
    'hyperparams': {
        'teacher': 'tf_efficientnetv2_s.in21k_ft_in1k',
        'student': 'mobilenet_v3_large',
        'kd_alpha': ALPHA, 'kd_tau': T_KD, 'la_tau': LA_TAU,
        'label_smoothing': LABEL_SMOOTHING,
        'use_weighted_sampler': bool(USE_WEIGHTED_SAMPLER),
        'use_source_weighted_sampler': bool(USE_SOURCE_WEIGHTED_SAMPLER),
        'source_weights': dict(SOURCE_WEIGHTS) if USE_SOURCE_WEIGHTED_SAMPLER else None,
        'batch': BATCH, 'img_size': IMG,
        'ema_decay': EMA_DECAY,
    },
    'int8': {
        '_note': ('f1_int8 / f1_fp32 are measured by raw argmax(logits) on '
                  'val_realworld — no temperature scaling, no weak-healthy '
                  'reroute, no very-low-confidence demotion. The runtime '
                  'pipeline applies all three on top; production disease '
                  'macro-F1 is therefore at least as good as f1_int8, never '
                  'less. The ship gate uses f1_int8 as a conservative floor.'),
        'drift': drift, 'f1_fp32': f1_fp32, 'f1_int8': f1_int8,
        'warn_drift': WARN_DRIFT, 'fail_drift': FAIL_DRIFT,
        'quant_mode': winner.get('quant_mode', 'unknown'),
        'activation_type': 'QUInt8', 'weight_type': 'QInt8',
        'per_channel': True, 'format': 'QDQ',
        'calibration_split': 'temperature_cal+train_backfill',
        'calibration_n': CALIB_TARGET_N,
        'static_extra_options': STATIC_QUANT_EXTRA_OPTIONS,
        'nodes_to_exclude': winner.get('nodes_to_exclude', []),
        'shipped': int8_shipped,
        'shipped_artifact': shipped_artifact,
    },
    'kd': {'best_indomain_disease_f1': best_f1, 'best_epoch': best_ep},
    'reporting_backend': ('onnxruntime_int8_cpu' if int8_shipped
                          else 'onnxruntime_fp32_cpu'),
    'reproducibility': {
        'seed': SEED,
        'torch': torch.__version__,
        'onnxruntime': ort.__version__,
        'cudnn_deterministic': bool(getattr(torch.backends.cudnn, 'deterministic', False)),
        'cudnn_benchmark':     bool(getattr(torch.backends.cudnn, 'benchmark', True)),
    },
}

# Ship gate — runs BEFORE we persist anything declarative.
# - MIN_DISEASE_F1 / MAX_HEALTHY_BIAS / zero_recall: catch the obvious
#   failure modes.
# - MIN_HEALTHY_RECALL: catches the inverse of healthy_bias — a model
#   that never predicts `healthy` and labels every plant as some disease
#   still passes disease_macro_f1, but ships false positives.
# Fix 3: staged v1 gate. 0.70 macro-F1 is not reachable with the current
# source mix (studio-trained, field-graded); ship v1 at 0.58 with an explicit
# plan to raise it as real-world indoor spot-disease images are collected. The
# zero-recall / healthy-bias / healthy-recall guards below stay strict so a
# degenerate model still fails loud.
MIN_DISEASE_F1, MAX_HEALTHY_BIAS, MIN_HEALTHY_RECALL = 0.58, 1.50, 0.70
rw = final_metrics['realworld']
disease_classes = [c for c in CLASS_NAMES if c not in ('healthy', 'not_a_plant')]
zero_recall = [c for c in disease_classes if rw['per_class_recall'].get(c, 0.0) <= 0.0]
healthy_recall = float(rw['per_class_recall'].get('healthy', 0.0))
failures = []
if rw['disease_macro_f1'] < MIN_DISEASE_F1:
    failures.append(f'disease_macro_f1={rw["disease_macro_f1"]:.3f} < {MIN_DISEASE_F1}')
if rw['healthy_bias'] >= MAX_HEALTHY_BIAS:
    failures.append(f'healthy_bias={rw["healthy_bias"]:.2f} >= {MAX_HEALTHY_BIAS}')
if healthy_recall < MIN_HEALTHY_RECALL:
    failures.append(f'recall[healthy]={healthy_recall:.3f} < {MIN_HEALTHY_RECALL} (false-positive disease risk)')
if zero_recall:
    failures.append(f'zero recall classes: {zero_recall}')
if failures:
    # Gate failed — don't persist the declarative artefacts. classifier.pth
    # and the canonical ONNX files have already been written above (they
    # are the cheapest things to overwrite next run), but classes.json /
    # temperature.json / conformal.json / training_metrics.json /
    # model_card.json stay un-updated so a downstream consumer reads the
    # last successful build.
    raise RuntimeError('SHIP GATE FAILED: ' + '; '.join(failures))
print('SHIP GATE PASSED')

# Persist the declarative artefacts now that the gate is green.
Path('models/classes.json').write_text(json.dumps(classes_payload, indent=2))
Path('models/temperature.json').write_text(json.dumps(temperature_payload, indent=2))
Path('models/conformal.json').write_text(json.dumps(cfg, indent=2))
Path('models/training_metrics.json').write_text(json.dumps(final_metrics, indent=2))
Path('models/model_card.json').write_text(json.dumps(model_card, indent=2))
print(f'classes.json: {len(CLASS_NAMES)} classes')
print('Persisted: classes.json, temperature.json, conformal.json, '
      'training_metrics.json, model_card.json')

# Detector artefact contract.
missing = [s for s in ('detector.pt', 'detector.onnx')
           if not Path(f'models/{s}').exists()]
if missing:
    print(f'WARN: {missing} not in models/ — drop the shipped '
          f'server/models/detector.{{pt,onnx}} pair into ./models/ '
          f'before assembling the deploy bundle, or re-run §3 with '
          f'a YOLO data.yaml under data_v2/.')

# Drop the runner-up KD state-dict bag unless an operator wants to keep
# it (e.g. for an offline re-quant experiment). The server never reads it.
KEEP_KD_CANDIDATES = False
if not KEEP_KD_CANDIDATES:
    candidates_path = Path('models/classifier_candidates.pt')
    if candidates_path.exists():
        candidates_path.unlink()
        print(f'cleaned up {candidates_path} (KEEP_KD_CANDIDATES=False)')

log_run({'phase': 'export_done', 'f1_fp32': f1_fp32, 'f1_int8': f1_int8, 'T': T_val,
         'q_hat': cfg['q_hat'], 'empirical_coverage': cfg['empirical_coverage'],
         'int8_shipped': int8_shipped, 'shipped_artifact': shipped_artifact})


## §7 — Evaluation contract

| Slice | Role | Pass criterion |
|---|---|---|
| `val_indomain` | training-loop metric, drives early stopping | tracked; best checkpoint restored |
| `val_realworld` | **primary — ship/no-ship gate** | disease macro-F1 ≥ **0.58** (staged v1); no disease class with recall = 0; healthy-bias < 1.5; healthy recall ≥ 0.70 |
| `val_studio` | PlantVillage transfer / background-bias monitor | tracked, never blocking |
| `temperature_cal` | INT8 activation calibration + temperature scaling | held out; backfilled from `train_ds` if thin |
| `conformal_cal` | RAPS calibration | held out; 80/20 fit/audit when n ≥ 200, else fit-only |

Persisted artefacts (all written to `models/` in the notebook CWD, **only after the ship gate passes**):

* `classifier.pth` — KD student weights (eager MobileNetV3-Large state_dict, no `_orig_mod.` prefix).
* `classifier.onnx` — winner FP32 export (also serves as fallback when INT8 fails the drift gate).
* `classifier_int8.onnx` — static-INT8 QDQ (or QOperator on retry). **Only present when INT8 drift ≤ 0.05.**
* `classifier_fp32.onnx` — copy of the FP32 graph. **Only present when INT8 drift > 0.05.** Production loader falls back to `classifier.onnx`; this file is the explicit "INT8 export failed" signal.
* `classes.json` — the contract the server reads at startup (`model_version`, class names, training hyperparams, OOD config).
* `temperature.json` — fitted T plus `source` ∈ {`int8`, `fp32_fallback`}.
* `conformal.json` — RAPS calibration; `fit_source` matches `temperature.json.source`.
* `detector.{pt,onnx}` — Stage 1 weights (re-trained from §3 if a YOLO `data.yaml` is supplied; the shipped pair otherwise).
* `dinov2_probe.json` — diagnostic linear-probe curve (cached-feature implementation).
* `training_metrics.json` — structured per-class recall + confusion matrix on every val slice.
* `model_card.json` — provenance: train/val counts per class, hyperparams, INT8 drift, best-KD epoch, reproducibility flags, `int8.shipped` flag and `shipped_artifact` label.
* `run_log.jsonl` — one JSON-line per epoch / phase, useful for post-mortem.
